# 05 - Pressure Analysis & Modeling

**Objective:** Quantifying pressure index for every ball and game state; validating high vs low pressure thresholds.

---
### supeRR Selector Pressure Analysis


In [1]:
# Setup and imports
import sys
from pathlib import Path

# Ensure src module is accessible
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.append(str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from superr_selector.data.loader import DataLoader

%matplotlib inline
print('Environment initialized successfully.')


Environment initialized successfully.


## 1. Load Datasets


In [2]:
# STEP 1: Prepare batter-level analysis dataset

batter_analysis_df = baseline_chase_df.copy()

# A batter faces a delivery only when they are the striker.
# Exclude wides because the striker does not face a legal ball.
extras_category = (
    batter_analysis_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

batter_analysis_df["is_batter_ball_faced"] = (
    extras_category != "wide"
).astype(int)

# Label each delivery as pressure or non-pressure
batter_analysis_df["performance_group"] = np.where(
    batter_analysis_df["primary_pressure"],
    "pressure",
    "baseline"
)

# Keep the fields needed for batter analysis
batter_analysis_df = batter_analysis_df[
    [
        "match_id",
        "match_format",
        "striker_id",
        "striker_name",
        "runs",
        "is_four",
        "is_six",
        "is_wicket",
        "dismissed_player_id",
        "is_batter_ball_faced",
        "performance_group",
        "primary_pressure"
    ]
].copy()

print("BATTER ANALYSIS DATASET")
print("Total delivery records:", len(batter_analysis_df))
print("Unique batters:", batter_analysis_df["striker_id"].nunique())
print("Unique matches:", batter_analysis_df["match_id"].nunique())

print("\nPerformance group distribution:")
display(
    batter_analysis_df["performance_group"]
    .value_counts()
    .to_frame("delivery_count")
)

print("\nFormat distribution:")
display(
    batter_analysis_df.groupby("match_format")
    .agg(
        deliveries=("match_id", "size"),
        batters=("striker_id", "nunique"),
        matches=("match_id", "nunique")
    )
)

print("\nMissing values:")
display(batter_analysis_df.isna().sum().to_frame("missing_values"))


NameError: name 'baseline_chase_df' is not defined

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

print("Current working directory:", PROJECT_ROOT)
print("\nFiles and folders:")
for item in sorted(PROJECT_ROOT.iterdir()):
    print(item.name)

Current working directory: d:\PROJECTS\supeRR-selector-pressure-analysis\notebooks

Files and folders:
01_data_understanding.ipynb
02_data_quality_analysis.ipynb
03_data_preprocessing.ipynb
04_feature_engineering.ipynb
05_pressure_analysis.ipynb
06_batter_performance_analysis.ipynb
07_bowler_performance_analysis.ipynb
08_player_ranking.ipynb
09_visualization.ipynb


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

for folder_name in ["data/raw", "data/interim", "data/processed"]:
    folder = PROJECT_ROOT / folder_name

    print(f"\n--- {folder_name} ---")

    if folder.exists():
        files = sorted(folder.iterdir())

        if files:
            for file in files:
                if file.is_file():
                    print(f"{file.name}  |  {file.stat().st_size / 1024:.2f} KB")
        else:
            print("Folder is empty.")
    else:
        print("Folder does not exist.")


--- data/raw ---
ball_by_ball.csv  |  40109.22 KB
competition_team_player_mapping.csv  |  452.12 KB
competitions.csv  |  7.45 KB
matches.csv  |  120.96 KB
playing_eleven.csv  |  943.64 KB
teams.csv  |  6.76 KB

--- data/interim ---
.gitkeep  |  0.00 KB

--- data/processed ---
.gitkeep  |  0.00 KB


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load required source datasets
ball_df = pd.read_csv(RAW_DIR / "ball_by_ball.csv")
matches_df = pd.read_csv(RAW_DIR / "matches.csv")
competitions_df = pd.read_csv(RAW_DIR / "competitions.csv")

print("DATASETS LOADED SUCCESSFULLY")
print("-" * 45)

for name, data in {
    "Ball-by-ball": ball_df,
    "Matches": matches_df,
    "Competitions": competitions_df
}.items():
    print(f"{name}: {data.shape}")

print("\nBall-by-ball columns:")
print(ball_df.columns.tolist())

print("\nCompetition formats:")
display(competitions_df["match_format"].value_counts(dropna=False))

DATASETS LOADED SUCCESSFULLY
---------------------------------------------
Ball-by-ball: (200999, 33)
Matches: (883, 22)
Competitions: (141, 4)

Ball-by-ball columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']

Competition formats:


match_format
T20I    134
T20       7
Name: count, dtype: int64

In [6]:
# STEP 4: Attach match and competition metadata

ball_df = ball_df.merge(
    matches_df[["match_id", "comp_id", "result"]],
    on="match_id",
    how="left",
    validate="many_to_one"
)

ball_df = ball_df.merge(
    competitions_df[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

# Identify IPL competitions within T20
is_ipl = (
    ball_df["match_format"].eq("T20")
    & ball_df["comp_name"].str.contains(
        "Indian Premier League|IPL",
        case=False,
        na=False,
        regex=True
    )
)

is_t20i = ball_df["match_format"].eq("T20I")

ball_df["primary_analysis_scope"] = is_ipl | is_t20i

print("MATCH AND COMPETITION METADATA ATTACHED")
print("-" * 45)

print("Total delivery records:", len(ball_df))
print("Missing competition format:", ball_df["match_format"].isna().sum())
print("Missing match result:", ball_df["result"].isna().sum())

print("\nPrimary analysis scope:")
display(
    ball_df.loc[ball_df["primary_analysis_scope"]]
    .groupby(["match_format", "comp_name"])
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

print("\nAll competition formats:")
display(
    ball_df.groupby("match_format")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique")
    )
)

MATCH AND COMPETITION METADATA ATTACHED
---------------------------------------------
Total delivery records: 200999
Missing competition format: 0
Missing match result: 0

Primary analysis scope:


,match_format,comp_name,deliveries,matches
0,T20,IPL 2022,17909,74
1,T20,IPL 2023,17863,74
2,T20,IPL 2024,17103,71
3,T20,IPL 2025,17285,74
4,T20,IPL 2026,17527,74
...,...,...,...,...
131,T20I,West Indies in South Africa T20I Series 2023,646,3
132,T20I,West Indies in South Africa T20I Series 2026,596,3
133,T20I,West Indies in Sri Lanka T20I Series 2024,705,3
134,T20I,Zimbabwe in Bangladesh T20I Series 2024,1198,5



All competition formats:


,deliveries,matches
match_format,,
T20,94672,397
T20I,106327,470


In [7]:
# STEP 5: Recreate validated innings eligibility rules

innings_eligibility = (
    ball_df.groupby(["match_id", "innings"])
    .agg(
        result=("result", "first"),
        match_format=("match_format", "first"),
        primary_analysis_scope=("primary_analysis_scope", "first"),
        recorded_deliveries=("innings_ball_num", "size")
    )
    .reset_index()
)

# Rule 1: Only regulation innings (1 and 2)
innings_eligibility["is_regulation_innings"] = (
    innings_eligibility["innings"].isin([1, 2])
)

# Rule 2: Exclude abandoned and no-result matches
innings_eligibility["is_completed_match_result"] = (
    ~innings_eligibility["result"]
    .fillna("")
    .str.contains(r"no result|abandoned", case=False, regex=True)
)

# Rule 3: Match 18532 second innings has an unverified chase target
innings_eligibility["has_reliable_chase_target"] = ~(
    (innings_eligibility["match_id"] == 18532)
    & (innings_eligibility["innings"] == 2)
)

# Final eligibility
innings_eligibility["primary_analysis_eligible"] = (
    innings_eligibility["is_regulation_innings"]
    & innings_eligibility["is_completed_match_result"]
    & innings_eligibility["has_reliable_chase_target"]
)

# Attach eligibility to every delivery
ball_df = ball_df.merge(
    innings_eligibility[
        ["match_id", "innings", "primary_analysis_eligible"]
    ],
    on=["match_id", "innings"],
    how="left",
    validate="many_to_one"
)

eligible_df = ball_df.loc[
    ball_df["primary_analysis_eligible"]
].copy()

# Restrict to IPL + T20I for our actual study
primary_eligible_df = eligible_df.loc[
    eligible_df["primary_analysis_scope"]
].copy()

print("INNINGS ELIGIBILITY AUDIT")
print("-" * 45)

print("Total innings:", len(innings_eligibility))
print("Eligible innings:", innings_eligibility["primary_analysis_eligible"].sum())
print("Excluded innings:", (~innings_eligibility["primary_analysis_eligible"]).sum())

print("\nEligible delivery records across all formats:", len(eligible_df))
print("Eligible delivery records in IPL + T20I:", len(primary_eligible_df))

print("\nPrimary study scope:")
display(
    primary_eligible_df.groupby("match_format")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique")
    )
)

print("\nExclusion reasons:")
display(
    innings_eligibility.loc[
        ~innings_eligibility["primary_analysis_eligible"]
    ].groupby(
        ["is_regulation_innings", "is_completed_match_result",
         "has_reliable_chase_target"]
    ).size().to_frame("excluded_innings_count")
)

INNINGS ELIGIBILITY AUDIT
---------------------------------------------
Total innings: 1724
Eligible innings: 1685
Excluded innings: 39

Eligible delivery records across all formats: 199152
Eligible delivery records in IPL + T20I: 192179

Primary study scope:


,deliveries,matches
match_format,,
T20,87217,362
T20I,104962,451



Exclusion reasons:


excluded_innings_count
is_regulation_innings is_completed_match_result has_reliable_chase_target                        
False                 True                      True                                           12
True                  False                     True                                           26
                      True                      False                                           1

In [8]:
# STEP 6: Reconstruct legal-ball classification and innings score

# Normalize extras categories
category = (
    primary_eligible_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

has_extras = (
    primary_eligible_df["extras1_runs"].fillna(0)
    + primary_eligible_df["extras_2_runs"].fillna(0)
) > 0

# Conservative legal-ball classification
primary_eligible_df["legal_status"] = "unresolved"

# Confirmed illegal deliveries
primary_eligible_df.loc[
    category.isin(["wide", "overstep"]),
    "legal_status"
] = "illegal"

# Confirmed legal deliveries
primary_eligible_df.loc[
    category.isin(["legbyes", "byes"]),
    "legal_status"
] = "legal"

# Ordinary deliveries: assumed legal, not independently verified
ordinary_candidate = (category == "") & (~has_extras)

primary_eligible_df.loc[
    ordinary_candidate,
    "legal_status"
] = "legal_assumed"

# Create legal-ball lower and upper bounds
primary_eligible_df["legal_ball_min_flag"] = (
    primary_eligible_df["legal_status"]
    .isin(["legal", "legal_assumed"])
).astype(int)

primary_eligible_df["legal_ball_max_flag"] = (
    primary_eligible_df["legal_ball_min_flag"]
    + (primary_eligible_df["legal_status"] == "unresolved").astype(int)
)

# Total runs scored from each delivery
primary_eligible_df["total_delivery_runs"] = (
    primary_eligible_df["runs"]
    + primary_eligible_df["extras1_runs"]
    + primary_eligible_df["extras_2_runs"]
)

# Calculate cumulative legal balls before and after each delivery
group_cols = ["match_id", "innings"]

primary_eligible_df = primary_eligible_df.sort_values(
    group_cols + ["innings_ball_num"]
).copy()

primary_eligible_df["legal_balls_min_after"] = (
    primary_eligible_df.groupby(group_cols)["legal_ball_min_flag"].cumsum()
)

primary_eligible_df["legal_balls_max_after"] = (
    primary_eligible_df.groupby(group_cols)["legal_ball_max_flag"].cumsum()
)

primary_eligible_df["legal_balls_min_before"] = (
    primary_eligible_df["legal_balls_min_after"]
    - primary_eligible_df["legal_ball_min_flag"]
)

primary_eligible_df["legal_balls_max_before"] = (
    primary_eligible_df["legal_balls_max_after"]
    - primary_eligible_df["legal_ball_max_flag"]
)

# Calculate cumulative innings score before and after each delivery
primary_eligible_df["innings_score_after"] = (
    primary_eligible_df.groupby(group_cols)["total_delivery_runs"].cumsum()
)

primary_eligible_df["innings_score_before"] = (
    primary_eligible_df["innings_score_after"]
    - primary_eligible_df["total_delivery_runs"]
)

print("LEGAL-BALL CLASSIFICATION")
display(
    primary_eligible_df["legal_status"]
    .value_counts()
    .to_frame("delivery_count")
)

print("\nLEGAL-BALL UNCERTAINTY")
print(
    "Minimum legal balls:",
    primary_eligible_df["legal_ball_min_flag"].sum()
)
print(
    "Maximum legal balls:",
    primary_eligible_df["legal_ball_max_flag"].sum()
)
print(
    "Unresolved deliveries:",
    (primary_eligible_df["legal_status"] == "unresolved").sum()
)

print("\nSCORE VALIDATION")
innings_score_check = (
    primary_eligible_df.groupby(group_cols)
    .agg(
        calculated_total=("total_delivery_runs", "sum"),
        final_cumulative_score=("innings_score_after", "max")
    )
)

innings_score_check["score_matches"] = (
    innings_score_check["calculated_total"]
    == innings_score_check["final_cumulative_score"]
)

print("Innings checked:", len(innings_score_check))
print("All innings scores reconcile:", innings_score_check["score_matches"].all())

print("\nDelivery records:", len(primary_eligible_df))

LEGAL-BALL CLASSIFICATION


,delivery_count
legal_status,
legal_assumed,180806
illegal,7974
legal,3140
unresolved,259



LEGAL-BALL UNCERTAINTY
Minimum legal balls: 183946
Maximum legal balls: 184205
Unresolved deliveries: 259

SCORE VALIDATION
Innings checked: 1625
All innings scores reconcile: True

Delivery records: 192179


In [9]:
# STEP 7: Reconstruct match-level chase targets and ball limits

# Get innings scores from the validated delivery-level data
innings_totals = (
    primary_eligible_df.groupby(["match_id", "innings"])
    .agg(
        innings_score=("total_delivery_runs", "sum"),
        legal_balls_min=("legal_ball_min_flag", "sum"),
        legal_balls_max=("legal_ball_max_flag", "sum")
    )
    .reset_index()
)

# Separate first and second innings summaries
first_innings = (
    innings_totals.loc[innings_totals["innings"] == 1]
    [["match_id", "innings_score"]]
    .rename(columns={"innings_score": "first_innings_score"})
)

second_innings = (
    innings_totals.loc[innings_totals["innings"] == 2]
    [["match_id", "innings_score"]]
    .rename(columns={"innings_score": "second_innings_score"})
)

# Match metadata
match_metadata = matches_df[
    [
        "match_id",
        "result",
        "revised_target",
        "first_inn_revised_overs",
        "second_inn_revised_overs"
    ]
].copy()

chase_match_audit = (
    match_metadata
    .merge(first_innings, on="match_id", how="left", validate="one_to_one")
    .merge(second_innings, on="match_id", how="left", validate="one_to_one")
)

# Restrict to matches in our primary eligible dataset
primary_match_ids = primary_eligible_df["match_id"].unique()

chase_match_audit = chase_match_audit.loc[
    chase_match_audit["match_id"].isin(primary_match_ids)
].copy()

# Identify DLS matches
chase_match_audit["is_dls_match"] = (
    chase_match_audit["result"]
    .fillna("")
    .str.contains(r"D/L|DLS", case=False, regex=True)
)

# Reconstruct target
chase_match_audit["chase_target"] = np.nan
chase_match_audit["target_source"] = "unknown"

has_revised_target = chase_match_audit["revised_target"].notna()

ordinary_match = (
    ~chase_match_audit["is_dls_match"]
    & ~has_revised_target
    & chase_match_audit["first_innings_score"].notna()
)

chase_match_audit.loc[
    has_revised_target, "chase_target"
] = chase_match_audit.loc[has_revised_target, "revised_target"]

chase_match_audit.loc[
    has_revised_target, "target_source"
] = "recorded_revised_target"

chase_match_audit.loc[
    ordinary_match, "chase_target"
] = (
    chase_match_audit.loc[ordinary_match, "first_innings_score"] + 1
)

chase_match_audit.loc[
    ordinary_match, "target_source"
] = "first_innings_score_plus_one"

# Reconstruct scheduled legal balls
chase_match_audit["scheduled_legal_balls"] = np.nan
chase_match_audit["scheduled_balls_source"] = "unknown"

has_revised_limit = (
    chase_match_audit["second_inn_revised_overs"].notna()
)

ordinary_full_length = (
    ~chase_match_audit["is_dls_match"]
    & ~has_revised_limit
)

chase_match_audit.loc[
    has_revised_limit, "scheduled_legal_balls"
] = chase_match_audit.loc[
    has_revised_limit, "second_inn_revised_overs"
]

chase_match_audit.loc[
    has_revised_limit, "scheduled_balls_source"
] = "recorded_revised_limit"

chase_match_audit.loc[
    ordinary_full_length, "scheduled_legal_balls"
] = 120

chase_match_audit.loc[
    ordinary_full_length, "scheduled_balls_source"
] = "standard_120_ball_assumption"

# Verification flags
chase_match_audit["target_verified"] = (
    chase_match_audit["chase_target"].notna()
)

chase_match_audit["scheduled_balls_verified"] = (
    chase_match_audit["scheduled_legal_balls"].notna()
)

print("CHASE TARGET AUDIT")
print("-" * 45)

print("Primary matches:", len(chase_match_audit))

print("\nTarget source:")
display(chase_match_audit["target_source"].value_counts().to_frame("matches"))

print("\nScheduled-ball source:")
display(
    chase_match_audit["scheduled_balls_source"]
    .value_counts()
    .to_frame("matches")
)

print("\nVerified target and schedule:")
print(
    "Both verified:",
    (
        chase_match_audit["target_verified"]
        & chase_match_audit["scheduled_balls_verified"]
    ).sum()
)

print("\nUnverified matches:")
display(
    chase_match_audit.loc[
        ~(
            chase_match_audit["target_verified"]
            & chase_match_audit["scheduled_balls_verified"]
        ),
        [
            "match_id",
            "result",
            "first_innings_score",
            "revised_target",
            "second_inn_revised_overs",
            "target_source",
            "scheduled_balls_source"
        ]
    ]
)

CHASE TARGET AUDIT
---------------------------------------------
Primary matches: 813

Target source:


,matches
target_source,
first_innings_score_plus_one,787
recorded_revised_target,22
unknown,4



Scheduled-ball source:


,matches
scheduled_balls_source,
standard_120_ball_assumption,787
recorded_revised_limit,22
unknown,4



Verified target and schedule:
Both verified: 809

Unverified matches:


,match_id,result,first_innings_score,revised_target,second_inn_revised_overs,target_source,scheduled_balls_source
190,20182,Punjab Kings Won by 7 Runs(D/L Method),191.0,NaN,NaN,unknown,unknown
284,23674,Bangladesh Won by 6 Wickets(D/L Method),116.0,NaN,NaN,unknown,unknown
293,24822,India Won by 2 Runs (DLS method),139.0,NaN,NaN,unknown,unknown
316,28334,New Zealand won by 17runs (DLS Method),110.0,NaN,NaN,unknown,unknown


In [10]:
# STEP 8: Reconstruct delivery-level chase context

chase_context = primary_eligible_df.loc[
    primary_eligible_df["innings"] == 2
].copy()

chase_metadata = chase_match_audit[
    [
        "match_id",
        "chase_target",
        "target_source",
        "scheduled_legal_balls",
        "scheduled_balls_source",
        "target_verified",
        "scheduled_balls_verified"
    ]
].copy()

chase_context = chase_context.merge(
    chase_metadata,
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Verify target and scheduled-ball information
target_ok = (
    chase_context["target_verified"]
    .astype("boolean")
    .fillna(False)
)

schedule_ok = (
    chase_context["scheduled_balls_verified"]
    .astype("boolean")
    .fillna(False)
)

chase_context["pressure_context_eligible"] = (
    target_ok & schedule_ok
).astype(bool)

# Wickets lost before the current delivery
chase_context["wickets_lost_before"] = (
    chase_context.groupby(["match_id", "innings"])["is_wicket"]
    .transform(lambda s: s.astype(int).cumsum() - s.astype(int))
)

# Runs required before the current delivery
chase_context["runs_required"] = (
    chase_context["chase_target"]
    - chase_context["innings_score_before"]
)

# Remaining balls as a range
chase_context["balls_remaining_min"] = (
    chase_context["scheduled_legal_balls"]
    - chase_context["legal_balls_max_before"]
)

chase_context["balls_remaining_max"] = (
    chase_context["scheduled_legal_balls"]
    - chase_context["legal_balls_min_before"]
)

# Required run-rate bounds
chase_context["required_run_rate_min"] = np.nan
chase_context["required_run_rate_max"] = np.nan

valid_rate_denominator = (
    chase_context["pressure_context_eligible"]
    & (chase_context["balls_remaining_min"] > 0)
    & (chase_context["balls_remaining_max"] > 0)
    & (chase_context["runs_required"] > 0)
)

# Lower RRR uses the maximum possible remaining balls
chase_context.loc[
    valid_rate_denominator, "required_run_rate_min"
] = (
    chase_context.loc[valid_rate_denominator, "runs_required"] * 6
    / chase_context.loc[valid_rate_denominator, "balls_remaining_max"]
)

# Upper RRR uses the minimum possible remaining balls
chase_context.loc[
    valid_rate_denominator, "required_run_rate_max"
] = (
    chase_context.loc[valid_rate_denominator, "runs_required"] * 6
    / chase_context.loc[valid_rate_denominator, "balls_remaining_min"]
)

# Valid chase states
chase_context["valid_chase_state"] = (
    valid_rate_denominator
    & (chase_context["wickets_lost_before"] < 10)
)

# Use the same conservative RRR definition as the original analysis
chase_context["rrr_for_classification"] = (
    chase_context["required_run_rate_min"]
)

print("CHASE CONTEXT AUDIT")
print("-" * 45)

print("Second-innings deliveries:", len(chase_context))
print(
    "Verified target and schedule:",
    chase_context["pressure_context_eligible"].sum()
)
print(
    "Valid positive chase states:",
    chase_context["valid_chase_state"].sum()
)

print("\nUnverified context deliveries:")
print(
    (~chase_context["pressure_context_eligible"]).sum()
)

print("\nValid chase-state summary:")
display(
    chase_context.loc[chase_context["valid_chase_state"]][
        [
            "runs_required",
            "wickets_lost_before",
            "balls_remaining_min",
            "balls_remaining_max",
            "required_run_rate_min",
            "required_run_rate_max"
        ]
    ].describe().round(2)
)

CHASE CONTEXT AUDIT
---------------------------------------------
Second-innings deliveries: 91846
Verified target and schedule: 91508
Valid positive chase states: 91453

Unverified context deliveries:
338

Valid chase-state summary:


,runs_required,wickets_lost_before,balls_remaining_min,balls_remaining_max,required_run_rate_min,required_run_rate_max
count,91453.00,91453.00,91453.00,91453.00,91453.00,91453.00
mean,102.60,2.71,64.13,64.19,11.37,11.47
std,54.87,2.29,33.15,33.11,13.01,13.76
min,1.00,0.00,1.00,1.00,0.07,0.07
25%,59.00,1.00,36.00,36.00,7.70,7.71
50%,101.00,2.00,65.00,65.00,9.67,9.68
75%,143.00,4.00,93.00,93.00,12.00,12.00
max,305.00,9.00,120.00,120.00,810.00,810.00


In [11]:
# STEP 9: Build competitive chase pool and classify pressure

valid_chases = chase_context.loc[
    chase_context["valid_chase_state"]
].copy()

# Exclude mathematically unreachable chase states
valid_chases["within_six_per_ball_ceiling"] = (
    valid_chases["runs_required"]
    <= valid_chases["balls_remaining_min"] * 6
)

pressure_pool = valid_chases.loc[
    valid_chases["within_six_per_ball_ceiling"]
].copy()

# Final primary pressure definition
pressure_pool["primary_pressure"] = (
    (pressure_pool["rrr_for_classification"] >= 15)
    & (pressure_pool["wickets_lost_before"] >= 3)
    & (pressure_pool["balls_remaining_min"] <= 60)
)

# Validate pressure classification by format
pressure_summary = (
    pressure_pool.groupby("match_format")
    .agg(
        competitive_chase_deliveries=("match_id", "size"),
        competitive_matches=("match_id", "nunique"),
        pressure_deliveries=("primary_pressure", "sum")
    )
    .reset_index()
)

pressure_summary["non_pressure_deliveries"] = (
    pressure_summary["competitive_chase_deliveries"]
    - pressure_summary["pressure_deliveries"]
)

print("COMPETITIVE CHASE POOL")
print("-" * 45)

print("Valid chase states:", len(valid_chases))
print("Competitive chase deliveries:", len(pressure_pool))
print(
    "Excluded mathematically unreachable states:",
    len(valid_chases) - len(pressure_pool)
)

print("\nPressure classification by format:")
display(pressure_summary)

print("\nTOTAL VALIDATION")
print("Pressure deliveries:", int(pressure_pool["primary_pressure"].sum()))
print("Pressure matches:", pressure_pool.loc[
    pressure_pool["primary_pressure"], "match_id"
].nunique())

print("\nExpected pressure deliveries: 8,558")
print(
    "Pressure count matches expected:",
    int(pressure_pool["primary_pressure"].sum()) == 8558
)

COMPETITIVE CHASE POOL
---------------------------------------------
Valid chase states: 91453
Competitive chase deliveries: 89850
Excluded mathematically unreachable states: 1603

Pressure classification by format:


,match_format,competitive_chase_deliveries,competitive_matches,pressure_deliveries,non_pressure_deliveries
0,T20,41194,361,3990,37204
1,T20I,48656,447,4568,44088



TOTAL VALIDATION
Pressure deliveries: 8558
Pressure matches: 414

Expected pressure deliveries: 8,558
Pressure count matches expected: True


In [12]:
# STEP 10: Create batter-level pressure and baseline statistics

batter_delivery_df = pressure_pool.copy()

# Normalize extras category
category = (
    batter_delivery_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# A wide is not counted as a ball faced.
# No-balls are counted as balls faced.
batter_delivery_df["batter_ball_faced"] = (
    category != "wide"
).astype(int)

# Attribute dismissals to the actual dismissed batter
batter_delivery_df["batter_dismissal"] = (
    batter_delivery_df["is_wicket"].eq(True)
    & batter_delivery_df["dismissed_player_id"].notna()
    & (
        batter_delivery_df["dismissed_player_id"]
        == batter_delivery_df["striker_id"]
    )
).astype(int)

# Separate pressure and baseline deliveries
batter_delivery_df["performance_group"] = np.where(
    batter_delivery_df["primary_pressure"],
    "pressure",
    "baseline"
)

# Aggregate player performance
batter_stats = (
    batter_delivery_df.groupby(
        [
            "match_format",
            "striker_id",
            "striker_name",
            "performance_group"
        ]
    )
    .agg(
        deliveries=("match_id", "size"),
        balls_faced=("batter_ball_faced", "sum"),
        runs=("runs", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("batter_dismissal", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
    .rename(
        columns={
            "striker_id": "batter_id",
            "striker_name": "batter_name"
        }
    )
)

# Calculate strike rate
batter_stats["strike_rate"] = np.where(
    batter_stats["balls_faced"] > 0,
    batter_stats["runs"] * 100 / batter_stats["balls_faced"],
    np.nan
)

print("BATTER PERFORMANCE DATASET")
print("-" * 50)

print("Player-group records:", len(batter_stats))
print("Unique batters:", batter_stats["batter_id"].nunique())

print("\nPerformance groups:")
display(
    batter_stats.groupby(
        ["match_format", "performance_group"]
    )
    .agg(
        batters=("batter_id", "nunique"),
        runs=("runs", "sum"),
        balls_faced=("balls_faced", "sum"),
        dismissals=("dismissals", "sum"),
        matches=("matches", "max")
    )
)

print("\nSample batter records:")
display(batter_stats.head(10))

print("\nMissing values:")
display(batter_stats.isna().sum().to_frame("missing_values"))

BATTER PERFORMANCE DATASET
--------------------------------------------------
Player-group records: 1343
Unique batters: 733

Performance groups:


batters   runs  balls_faced  dismissals  \
match_format performance_group                                            
T20          baseline               265  51761        35836        1705   
             pressure               209   5997         3733         314   
T20I         baseline               531  55863        42535        2324   
             pressure               338   6166         4356         412   

                                matches  
match_format performance_group           
T20          baseline                42  
             pressure                14  
T20I         baseline                36  
             pressure                13


Sample batter records:


,match_format,batter_id,batter_name,performance_group,deliveries,balls_faced,runs,fours,sixes,dismissals,matches,strike_rate
0,T20,8,Mahendra Singh Dhoni,baseline,92,88,86,5,2,1,13,97.727273
1,T20,8,Mahendra Singh Dhoni,pressure,115,103,176,11,13,6,13,170.873786
2,T20,23,Piyush Chawla,baseline,2,2,1,0,0,0,1,50.000000
3,T20,23,Piyush Chawla,pressure,18,18,24,1,2,4,5,133.333333
4,T20,28,Ishant Sharma,pressure,6,6,1,0,0,0,2,16.666667
5,T20,47,Virat Kohli,baseline,993,960,1473,155,51,24,34,153.437500
6,T20,88,Faf du Plessis,baseline,483,475,732,75,35,24,24,154.105263
7,T20,109,Wriddhiman Saha,baseline,382,372,479,60,10,18,20,128.763441
8,T20,126,Rohit Sharma,baseline,811,776,1112,99,67,37,40,143.298969
9,T20,126,Rohit Sharma,pressure,13,11,19,1,1,0,1,172.727273



Missing values:


,missing_values
match_format,0
batter_id,0
batter_name,0
performance_group,0
deliveries,0
balls_faced,0
runs,0
fours,0
sixes,0
dismissals,0


In [13]:
# STEP 11: Audit batter dismissal attribution

dismissal_rows = batter_delivery_df.loc[
    batter_delivery_df["is_wicket"].eq(True)
].copy()

dismissal_rows["dismissal_attribution"] = np.select(
    [
        dismissal_rows["dismissed_player_id"].isna(),
        dismissal_rows["dismissed_player_id"] == dismissal_rows["striker_id"],
        dismissal_rows["dismissed_player_id"] == dismissal_rows["non_striker_id"]
    ],
    [
        "missing_dismissed_player",
        "striker_dismissed",
        "non_striker_dismissed"
    ],
    default="other_or_unmatched"
)

print("DISMISSAL ATTRIBUTION AUDIT")
print("-" * 45)

display(
    dismissal_rows["dismissal_attribution"]
    .value_counts(dropna=False)
    .to_frame("delivery_count")
)

print("\nDismissal types:")
display(
    dismissal_rows["dismissal_type"]
    .value_counts(dropna=False)
    .to_frame("delivery_count")
)

print("\nDismissals by performance group:")
display(
    dismissal_rows.groupby(
        ["match_format", "performance_group", "dismissal_attribution"],
        dropna=False
    ).size().to_frame("dismissals")
)

print("\nBATTER SAMPLE AUDIT")

print(
    "Pressure deliveries:",
    int(batter_delivery_df["primary_pressure"].sum())
)

print(
    "Unique strikers in pressure:",
    batter_delivery_df.loc[
        batter_delivery_df["primary_pressure"], "striker_id"
    ].nunique()
)

print(
    "Unique strikers in baseline:",
    batter_delivery_df.loc[
        ~batter_delivery_df["primary_pressure"], "striker_id"
    ].nunique()
)

print("\nRecords with zero balls faced:")
display(
    batter_stats.loc[
        batter_stats["balls_faced"] == 0,
        [
            "match_format", "batter_id", "batter_name",
            "performance_group", "deliveries", "runs",
            "dismissals", "matches"
        ]
    ]
)

print("\nAudit complete.")

DISMISSAL ATTRIBUTION AUDIT
---------------------------------------------


,delivery_count
dismissal_attribution,
striker_dismissed,4755
non_striker_dismissed,103
missing_dismissed_player,1



Dismissal types:


,delivery_count
dismissal_type,
Caught,3374
Bowled,816
LBW,333
RunOut,220
Stump,103
Hit Wicket,8
Retire Out,4
Obstracting the field,1



Dismissals by performance group:


dismissals
match_format performance_group dismissal_attribution               
T20          baseline          non_striker_dismissed             39
                               striker_dismissed               1705
             pressure          missing_dismissed_player           1
                               non_striker_dismissed             12
                               striker_dismissed                314
T20I         baseline          non_striker_dismissed             44
                               striker_dismissed               2324
             pressure          non_striker_dismissed              8
                               striker_dismissed                412


BATTER SAMPLE AUDIT
Pressure deliveries: 8558
Unique strikers in pressure: 451
Unique strikers in baseline: 660

Records with zero balls faced:


,match_format,batter_id,batter_name,performance_group,deliveries,runs,dismissals,matches
1257,T20I,27743,DT Doram,pressure,1,0,0,1



Audit complete.


In [14]:
# STEP 12: Correct batter dismissal attribution

# Keep wicket deliveries where the dismissed player's ID is available
player_dismissal_rows = batter_delivery_df.loc[
    batter_delivery_df["is_wicket"].eq(True)
    & batter_delivery_df["dismissed_player_id"].notna()
].copy()

# Attribute each dismissal to the actual dismissed player
corrected_dismissal_stats = (
    player_dismissal_rows
    .groupby(
        ["match_format", "dismissed_player_id", "performance_group"],
        as_index=False
    )
    .agg(
        dismissals=("is_wicket", "size")
    )
    .rename(columns={"dismissed_player_id": "batter_id"})
)

# Preserve the original statistics for comparison
old_dismissal_total = batter_stats["dismissals"].sum()

# Remove the old striker-only dismissal counts
batter_stats = batter_stats.drop(columns=["dismissals"])

# Merge corrected dismissal counts into batter statistics
batter_stats = batter_stats.merge(
    corrected_dismissal_stats,
    on=["match_format", "batter_id", "performance_group"],
    how="left",
    validate="one_to_one"
)

batter_stats["dismissals"] = (
    batter_stats["dismissals"].fillna(0).astype(int)
)

# Recalculate batting average using corrected dismissals
batter_stats["batting_average"] = np.where(
    batter_stats["dismissals"] > 0,
    batter_stats["runs"] / batter_stats["dismissals"],
    np.nan
)

print("OLD STRIKER-ONLY DISMISSAL TOTAL:", old_dismissal_total)
print("CORRECTED PLAYER-ATTRIBUTED DISMISSALS:",
      batter_stats["dismissals"].sum())
print("UNATTRIBUTED WICKET DELIVERIES:",
      batter_delivery_df.loc[
          batter_delivery_df["is_wicket"].eq(True)
          & batter_delivery_df["dismissed_player_id"].isna()
      ].shape[0])

print("\nCORRECTED DISMISSALS BY FORMAT AND PERFORMANCE GROUP")

display(
    batter_stats.groupby(
        ["match_format", "performance_group"],
        as_index=False
    ).agg(
        batters=("batter_id", "nunique"),
        runs=("runs", "sum"),
        balls_faced=("balls_faced", "sum"),
        dismissals=("dismissals", "sum"),
        batter_records=("batter_id", "size")
    )
)

print("\nSAMPLE BATTER RECORDS")
display(
    batter_stats.sort_values(
        ["match_format", "performance_group", "runs"],
        ascending=[True, True, False]
    ).head(10)
)

OLD STRIKER-ONLY DISMISSAL TOTAL: 4755
CORRECTED PLAYER-ATTRIBUTED DISMISSALS: 4854
UNATTRIBUTED WICKET DELIVERIES: 1

CORRECTED DISMISSALS BY FORMAT AND PERFORMANCE GROUP


,match_format,performance_group,batters,runs,balls_faced,dismissals,batter_records
0,T20,baseline,265,51761,35836,1744,265
1,T20,pressure,209,5997,3733,325,209
2,T20I,baseline,531,55863,42535,2366,531
3,T20I,pressure,338,6166,4356,419,338



SAMPLE BATTER RECORDS


,match_format,batter_id,batter_name,performance_group,deliveries,balls_faced,runs,fours,sixes,matches,strike_rate,dismissals,batting_average
288,T20,6288,Shubman Gill,baseline,1133,1101,1590,158,58,42,144.414169,38,41.842105
5,T20,47,Virat Kohli,baseline,993,960,1473,155,51,34,153.437500,25,58.920000
330,T20,6819,Yashasvi Jaiswal,baseline,849,820,1286,151,53,36,156.829268,32,40.187500
90,T20,1699,Suryakumar Yadav,baseline,743,725,1247,131,59,34,172.000000,23,54.217391
412,T20,23450,Sai Sudharsan,baseline,848,822,1206,130,37,31,146.715328,26,46.384615
49,T20,813,Jos Buttler,baseline,755,735,1131,116,50,35,153.877551,28,40.392857
59,T20,953,Lokesh Rahul,baseline,831,811,1123,97,47,33,138.471023,30,37.433333
8,T20,126,Rohit Sharma,baseline,811,776,1112,99,67,40,143.298969,37,30.054054
180,T20,3647,Shreyas Iyer,baseline,699,669,1107,91,62,29,165.470852,17,65.117647
202,T20,4117,Ishan Kishan,baseline,763,743,1077,116,49,37,144.952894,35,30.771429


In [15]:
# STEP 13: Reconciliation audit

# 1. Direct dismissal totals from delivery-level data
direct_dismissals = (
    batter_delivery_df.loc[
        batter_delivery_df["is_wicket"].eq(True)
        & batter_delivery_df["dismissed_player_id"].notna()
    ]
    .groupby(["match_format", "performance_group"])
    .size()
    .rename("direct_dismissals")
    .reset_index()
)

# 2. Dismissals successfully represented in batter_stats
merged_dismissals = (
    batter_stats
    .groupby(["match_format", "performance_group"])["dismissals"]
    .sum()
    .rename("merged_dismissals")
    .reset_index()
)

reconciliation = direct_dismissals.merge(
    merged_dismissals,
    on=["match_format", "performance_group"],
    how="outer"
).fillna(0)

reconciliation["difference"] = (
    reconciliation["direct_dismissals"]
    - reconciliation["merged_dismissals"]
)

print("DISMISSAL RECONCILIATION")
display(reconciliation)

# 3. Find dismissed-player records that did not match a batter_stats row
dismissal_keys = (
    player_dismissal_rows[
        ["match_format", "dismissed_player_id", "performance_group"]
    ]
    .drop_duplicates()
    .rename(columns={"dismissed_player_id": "batter_id"})
)

batter_keys = batter_stats[
    ["match_format", "batter_id", "performance_group"]
].drop_duplicates()

unmatched_dismissed_players = dismissal_keys.merge(
    batter_keys,
    on=["match_format", "batter_id", "performance_group"],
    how="left",
    indicator=True
)

unmatched_dismissed_players = unmatched_dismissed_players.loc[
    unmatched_dismissed_players["_merge"].eq("left_only")
]

print("\nDISMISSED PLAYERS WITHOUT A MATCHING BATTER STAT ROW:",
      len(unmatched_dismissed_players))
display(unmatched_dismissed_players.head(20))

# 4. Reconcile unique batter IDs from source deliveries and aggregated stats
print("\nUNIQUE BATTER ID AUDIT")

for fmt in ["T20", "T20I"]:
    source = batter_delivery_df.loc[
        batter_delivery_df["match_format"].eq(fmt)
    ]

    aggregated = batter_stats.loc[
        batter_stats["match_format"].eq(fmt)
    ]

    print(
        fmt,
        "| source unique strikers:", source["striker_id"].nunique(),
        "| aggregated unique batters:", aggregated["batter_id"].nunique(),
        "| pressure source:", source.loc[
            source["primary_pressure"], "striker_id"
        ].nunique(),
        "| pressure aggregated:", aggregated.loc[
            aggregated["performance_group"].eq("pressure"),
            "batter_id"
        ].nunique()
    )

DISMISSAL RECONCILIATION


,match_format,performance_group,direct_dismissals,merged_dismissals,difference
0,T20,baseline,1744,1744,0
1,T20,pressure,326,325,1
2,T20I,baseline,2368,2366,2
3,T20I,pressure,420,419,1



DISMISSED PLAYERS WITHOUT A MATCHING BATTER STAT ROW: 4


,match_format,batter_id,performance_group,_merge
365,T20I,6188.0,baseline,left_only
618,T20I,5808.0,pressure,left_only
647,T20,4716.0,pressure,left_only
708,T20I,8614.0,baseline,left_only



UNIQUE BATTER ID AUDIT
T20 | source unique strikers: 305 | aggregated unique batters: 305 | pressure source: 209 | pressure aggregated: 209
T20I | source unique strikers: 590 | aggregated unique batters: 590 | pressure source: 338 | pressure aggregated: 338


In [16]:
# STEP 14: Preserve dismissals for players with no batting exposure in that group

# Build a player ID -> name lookup from both striker and non-striker fields
player_names = pd.concat([
    primary_eligible_df[["striker_id", "striker_name"]].rename(
        columns={"striker_id": "batter_id", "striker_name": "batter_name"}
    ),
    primary_eligible_df[["non_striker_id", "non_striker_name"]].rename(
        columns={"non_striker_id": "batter_id", "non_striker_name": "batter_name"}
    )
]).dropna(subset=["batter_id", "batter_name"])

player_names = (
    player_names.drop_duplicates(subset=["batter_id"])
    .set_index("batter_id")["batter_name"]
)

# Count the missing players' dismissals in each format/group
missing_records = (
    player_dismissal_rows
    .merge(
        unmatched_dismissed_players[
            ["match_format", "batter_id", "performance_group"]
        ],
        left_on=["match_format", "dismissed_player_id", "performance_group"],
        right_on=["match_format", "batter_id", "performance_group"],
        how="inner"
    )
    .groupby(
        ["match_format", "batter_id", "performance_group"],
        as_index=False
    )
    .agg(dismissals=("is_wicket", "size"))
)

missing_records["batter_name"] = missing_records["batter_id"].map(player_names)

# Create rows with zero batting exposure, preserving their dismissal counts
for col in [
    "deliveries", "balls_faced", "runs", "fours", "sixes", "matches"
]:
    missing_records[col] = 0

missing_records["strike_rate"] = np.nan
missing_records["batting_average"] = np.nan

# Match the existing batter_stats column order
missing_records = missing_records[batter_stats.columns]

batter_stats = pd.concat(
    [batter_stats, missing_records],
    ignore_index=True
)

# Recalculate average for all records
batter_stats["batting_average"] = np.where(
    batter_stats["dismissals"] > 0,
    batter_stats["runs"] / batter_stats["dismissals"],
    np.nan
)

print("ADDED DISMISSAL-ONLY RECORDS:", len(missing_records))
display(missing_records)

print("\nFINAL DISMISSAL RECONCILIATION")
display(
    batter_stats.groupby(
        ["match_format", "performance_group"],
        as_index=False
    ).agg(
        dismissals=("dismissals", "sum"),
        zero_delivery_records=("deliveries", lambda x: (x == 0).sum()),
        zero_balls_faced_records=("balls_faced", lambda x: (x == 0).sum())
    )
)

print("\nTOTAL ATTRIBUTED DISMISSALS:", batter_stats["dismissals"].sum())
print(
    "DIRECT SOURCE DISMISSALS:",
    player_dismissal_rows.shape[0]
)

ADDED DISMISSAL-ONLY RECORDS: 4


,match_format,batter_id,batter_name,performance_group,deliveries,balls_faced,runs,fours,sixes,matches,strike_rate,dismissals,batting_average
0,T20,4716.0,Mayank Dagar,pressure,0,0,0,0,0,0,NaN,1,NaN
1,T20I,5808.0,Karim Janat,pressure,0,0,0,0,0,0,NaN,1,NaN
2,T20I,6188.0,Simi Singh,baseline,0,0,0,0,0,0,NaN,1,NaN
3,T20I,8614.0,William ORourke,baseline,0,0,0,0,0,0,NaN,1,NaN



FINAL DISMISSAL RECONCILIATION


,match_format,performance_group,dismissals,zero_delivery_records,zero_balls_faced_records
0,T20,baseline,1744,0,0
1,T20,pressure,326,1,1
2,T20I,baseline,2368,2,2
3,T20I,pressure,420,1,2



TOTAL ATTRIBUTED DISMISSALS: 4858
DIRECT SOURCE DISMISSALS: 4858


In [17]:
# STEP 15: Batter exposure and sample reliability audit

# Exclude dismissal-only records from batting exposure analysis
batting_exposure = batter_stats.loc[
    batter_stats["deliveries"] > 0
].copy()

# Summarise exposure distribution by format and performance group
exposure_summary = (
    batting_exposure
    .groupby(["match_format", "performance_group"])
    .agg(
        batters=("batter_id", "nunique"),
        total_deliveries=("deliveries", "sum"),
        total_balls_faced=("balls_faced", "sum"),
        median_deliveries=("deliveries", "median"),
        p25_deliveries=("deliveries", lambda x: x.quantile(0.25)),
        p75_deliveries=("deliveries", lambda x: x.quantile(0.75)),
        median_matches=("matches", "median"),
        min_matches=("matches", "min"),
        max_matches=("matches", "max")
    )
    .reset_index()
)

print("BATTER EXPOSURE DISTRIBUTION")
display(exposure_summary)

# Inspect pressure sample thresholds
pressure_exposure = batting_exposure.loc[
    batting_exposure["performance_group"].eq("pressure")
].copy()

threshold_audit = []

for fmt in ["T20", "T20I"]:
    fmt_data = pressure_exposure.loc[
        pressure_exposure["match_format"].eq(fmt)
    ]

    for min_deliveries in [12, 18, 24, 30, 40]:
        for min_matches in [3, 5, 7]:
            qualified = fmt_data.loc[
                (fmt_data["deliveries"] >= min_deliveries)
                & (fmt_data["matches"] >= min_matches)
                & (fmt_data["balls_faced"] > 0)
            ]

            threshold_audit.append({
                "match_format": fmt,
                "minimum_pressure_deliveries": min_deliveries,
                "minimum_pressure_matches": min_matches,
                "qualified_batters": qualified["batter_id"].nunique()
            })

threshold_audit = pd.DataFrame(threshold_audit)

print("\nPRESSURE SAMPLE THRESHOLD SENSITIVITY")
display(
    threshold_audit.pivot_table(
        index=["minimum_pressure_deliveries", "minimum_pressure_matches"],
        columns="match_format",
        values="qualified_batters"
    ).reset_index()
)

# Inspect the largest pressure samples to understand realistic exposure
print("\nTOP PRESSURE EXPOSURES")
display(
    pressure_exposure.sort_values(
        ["match_format", "deliveries", "matches"],
        ascending=[True, False, False]
    ).groupby("match_format").head(15)[
        [
            "match_format", "batter_name", "deliveries",
            "balls_faced", "matches", "runs", "dismissals"
        ]
    ]
)

BATTER EXPOSURE DISTRIBUTION


,match_format,performance_group,batters,total_deliveries,total_balls_faced,median_deliveries,p25_deliveries,p75_deliveries,median_matches,min_matches,max_matches
0,T20,baseline,265,37204,35836,44.0,10.0,197.0,5.0,1,42
1,T20,pressure,209,3990,3733,11.0,4.0,24.0,2.0,1,14
2,T20I,baseline,531,44088,42535,29.0,8.0,105.5,3.0,1,36
3,T20I,pressure,338,4568,4356,9.0,4.0,18.0,2.0,1,13



PRESSURE SAMPLE THRESHOLD SENSITIVITY


match_format,minimum_pressure_deliveries,minimum_pressure_matches,T20,T20I
0,12,3,74.0,83.0
1,12,5,48.0,38.0
2,12,7,22.0,13.0
3,18,3,61.0,64.0
4,18,5,48.0,34.0
5,18,7,22.0,13.0
6,24,3,51.0,51.0
7,24,5,42.0,29.0
8,24,7,22.0,11.0
9,30,3,41.0,31.0



TOP PRESSURE EXPOSURES


,match_format,batter_name,deliveries,balls_faced,matches,runs,dismissals
1,T20,Mahendra Singh Dhoni,115,103,13,176,6
231,T20,Rinku Singh,112,102,7,248,0
15,T20,Ravindra Jadeja,111,107,13,185,2
196,T20,Rahul Tewatia,93,90,14,180,6
136,T20,Shashank Singh,93,88,10,133,5
294,T20,Tim David,87,77,12,168,6
145,T20,Axar Patel,86,78,12,117,6
234,T20,Rashid Khan,82,74,12,158,5
351,T20,Abdul Samad,78,74,8,101,6
177,T20,Kuldeep Yadav,70,67,11,52,3


In [18]:
# STEP 16: Baseline sample reliability audit

# Provisional pressure eligibility: 18 deliveries across at least 5 matches
pressure_candidates = batting_exposure.loc[
    (batting_exposure["performance_group"] == "pressure")
    & (batting_exposure["deliveries"] >= 18)
    & (batting_exposure["matches"] >= 5)
    & (batting_exposure["balls_faced"] > 0),
    ["match_format", "batter_id", "batter_name",
     "deliveries", "balls_faced", "matches"]
].copy()

pressure_candidates = pressure_candidates.rename(columns={
    "deliveries": "pressure_deliveries",
    "balls_faced": "pressure_balls_faced",
    "matches": "pressure_matches"
})

# Get each candidate's baseline exposure
baseline_exposure = batting_exposure.loc[
    batting_exposure["performance_group"] == "baseline",
    ["match_format", "batter_id", "deliveries",
     "balls_faced", "matches", "runs", "dismissals"]
].rename(columns={
    "deliveries": "baseline_deliveries",
    "balls_faced": "baseline_balls_faced",
    "matches": "baseline_matches",
    "runs": "baseline_runs",
    "dismissals": "baseline_dismissals"
})

candidate_baseline_audit = pressure_candidates.merge(
    baseline_exposure,
    on=["match_format", "batter_id"],
    how="left",
    validate="one_to_one"
)

# Fill absent baseline records with zero exposure
baseline_columns = [
    "baseline_deliveries", "baseline_balls_faced",
    "baseline_matches", "baseline_runs", "baseline_dismissals"
]
candidate_baseline_audit[baseline_columns] = (
    candidate_baseline_audit[baseline_columns].fillna(0)
)

print("PRESSURE-QUALIFIED BATTERS:", len(pressure_candidates))

print("\nBASELINE EXPOSURE DISTRIBUTION FOR THESE CANDIDATES")
display(
    candidate_baseline_audit.groupby("match_format").agg(
        candidates=("batter_id", "nunique"),
        median_baseline_balls=("baseline_balls_faced", "median"),
        p25_baseline_balls=("baseline_balls_faced",
                            lambda x: x.quantile(0.25)),
        p75_baseline_balls=("baseline_balls_faced",
                            lambda x: x.quantile(0.75)),
        median_baseline_matches=("baseline_matches", "median"),
        minimum_baseline_balls=("baseline_balls_faced", "min"),
        minimum_baseline_matches=("baseline_matches", "min")
    ).reset_index()
)

# Test different baseline minimums
baseline_thresholds = []

for fmt in ["T20", "T20I"]:
    fmt_data = candidate_baseline_audit.loc[
        candidate_baseline_audit["match_format"] == fmt
    ]

    for min_balls in [30, 60, 100, 150, 200]:
        for min_matches in [3, 5, 8]:
            qualified = fmt_data.loc[
                (fmt_data["baseline_balls_faced"] >= min_balls)
                & (fmt_data["baseline_matches"] >= min_matches)
            ]

            baseline_thresholds.append({
                "match_format": fmt,
                "minimum_baseline_balls": min_balls,
                "minimum_baseline_matches": min_matches,
                "qualified_batters": qualified["batter_id"].nunique()
            })

baseline_thresholds = pd.DataFrame(baseline_thresholds)

print("\nBASELINE THRESHOLD SENSITIVITY")
display(
    baseline_thresholds.pivot_table(
        index=["minimum_baseline_balls", "minimum_baseline_matches"],
        columns="match_format",
        values="qualified_batters"
    ).reset_index()
)

print("\nCANDIDATES WITH THE SMALLEST BASELINE SAMPLES")
display(
    candidate_baseline_audit.sort_values(
        ["match_format", "baseline_balls_faced", "baseline_matches"]
    ).groupby("match_format").head(10)[
        [
            "match_format", "batter_name",
            "pressure_deliveries", "pressure_matches",
            "baseline_balls_faced", "baseline_matches"
        ]
    ]
)

PRESSURE-QUALIFIED BATTERS: 82

BASELINE EXPOSURE DISTRIBUTION FOR THESE CANDIDATES


,match_format,candidates,median_baseline_balls,p25_baseline_balls,p75_baseline_balls,median_baseline_matches,minimum_baseline_balls,minimum_baseline_matches
0,T20,48,163.5,31.50,290.25,16.0,1.0,1.0
1,T20I,34,46.5,11.75,119.75,7.0,0.0,0.0



BASELINE THRESHOLD SENSITIVITY


match_format,minimum_baseline_balls,minimum_baseline_matches,T20,T20I
0,30,3,38.0,19.0
1,30,5,38.0,19.0
2,30,8,34.0,15.0
3,60,3,33.0,13.0
4,60,5,33.0,13.0
5,60,8,33.0,12.0
6,100,3,29.0,10.0
7,100,5,29.0,10.0
8,100,8,29.0,10.0
9,150,3,26.0,7.0



CANDIDATES WITH THE SMALLEST BASELINE SAMPLES


,match_format,batter_name,pressure_deliveries,pressure_matches,baseline_balls_faced,baseline_matches
11,T20,Dushmantha Chameera,32,6,1.0,1.0
1,T20,Piyush Chawla,18,5,2.0,1.0
32,T20,Mohammed Siraj,37,7,4.0,1.0
18,T20,Jofra Archer,39,5,13.0,4.0
33,T20,Rahul Chahar,27,5,17.0,2.0
5,T20,Umesh Yadav,24,8,22.0,4.0
24,T20,Mitchell Santner,23,6,25.0,3.0
20,T20,Kagiso Rabada,32,5,26.0,5.0
38,T20,Romario Shepherd,38,6,26.0,6.0
21,T20,Kuldeep Yadav,70,11,29.0,9.0


In [19]:
# STEP 17: Final batter eligibility selection

MIN_PRESSURE_DELIVERIES = 18
MIN_PRESSURE_MATCHES = 5

MIN_BASELINE_BALLS = 60
MIN_BASELINE_MATCHES = 3

eligible_batters = candidate_baseline_audit.loc[
    (candidate_baseline_audit["pressure_deliveries"] >= MIN_PRESSURE_DELIVERIES)
    & (candidate_baseline_audit["pressure_matches"] >= MIN_PRESSURE_MATCHES)
    & (candidate_baseline_audit["pressure_balls_faced"] > 0)
    & (candidate_baseline_audit["baseline_balls_faced"] >= MIN_BASELINE_BALLS)
    & (candidate_baseline_audit["baseline_matches"] >= MIN_BASELINE_MATCHES)
].copy()

eligible_batters = eligible_batters.sort_values(
    ["match_format", "batter_name"]
).reset_index(drop=True)

print("FINAL ELIGIBLE BATTER COUNTS")

display(
    eligible_batters.groupby("match_format").agg(
        eligible_batters=("batter_id", "nunique"),
        total_pressure_deliveries=("pressure_deliveries", "sum"),
        total_pressure_matches=("pressure_matches", "sum"),
        median_pressure_deliveries=("pressure_deliveries", "median"),
        median_pressure_matches=("pressure_matches", "median"),
        median_baseline_balls=("baseline_balls_faced", "median"),
        median_baseline_matches=("baseline_matches", "median")
    ).reset_index()
)

print("\nELIGIBLE BATTER LIST")
display(
    eligible_batters[
        [
            "match_format", "batter_id", "batter_name",
            "pressure_deliveries", "pressure_balls_faced",
            "pressure_matches", "baseline_balls_faced",
            "baseline_matches"
        ]
    ]
)

print("\nTOTAL ELIGIBLE BATTERS:", eligible_batters["batter_id"].nunique())

FINAL ELIGIBLE BATTER COUNTS


,match_format,eligible_batters,total_pressure_deliveries,total_pressure_matches,median_pressure_deliveries,median_pressure_matches,median_baseline_balls,median_baseline_matches
0,T20,33,1922,255,52.0,7.0,252.0,20.0
1,T20I,13,760,95,60.0,6.0,150.0,17.0



ELIGIBLE BATTER LIST


,match_format,batter_id,batter_name,pressure_deliveries,pressure_balls_faced,pressure_matches,baseline_balls_faced,baseline_matches
0,T20,7446.0,Abdul Samad,78,74,8,97.0,18.0
1,T20,1318.0,Andre Russell,36,34,5,154.0,16.0
2,T20,8101.0,Ashutosh Sharma,35,30,5,173.0,13.0
3,T20,2876.0,Axar Patel,86,78,12,294.0,27.0
4,T20,6460.0,Ayush Badoni,52,49,5,270.0,24.0
5,T20,1077.0,David Miller,36,34,8,481.0,30.0
6,T20,7260.0,Dhruv Jurel,67,61,6,354.0,27.0
7,T20,143.0,Dinesh Karthik,70,57,7,132.0,16.0
8,T20,3330.0,Hardik Pandya,59,58,11,392.0,31.0
9,T20,4260.0,Heinrich Klaasen,48,47,5,349.0,20.0



TOTAL ELIGIBLE BATTERS: 42


In [20]:
# STEP 18: Pressure vs baseline batting performance metrics

# Keep only eligible batters
eligible_keys = eligible_batters[
    ["match_format", "batter_id", "batter_name"]
].drop_duplicates()

eligible_stats = batter_stats.merge(
    eligible_keys,
    on=["match_format", "batter_id", "batter_name"],
    how="inner",
    validate="many_to_one"
)

# Separate pressure and baseline statistics
pressure_metrics = eligible_stats.loc[
    eligible_stats["performance_group"].eq("pressure")
].copy()

baseline_metrics = eligible_stats.loc[
    eligible_stats["performance_group"].eq("baseline")
].copy()

metric_columns = [
    "deliveries", "balls_faced", "runs", "fours",
    "sixes", "dismissals", "matches"
]

pressure_metrics = pressure_metrics[
    ["match_format", "batter_id"] + metric_columns
].rename(columns={
    col: f"pressure_{col}" for col in metric_columns
})

baseline_metrics = baseline_metrics[
    ["match_format", "batter_id"] + metric_columns
].rename(columns={
    col: f"baseline_{col}" for col in metric_columns
})

batter_comparison = pressure_metrics.merge(
    baseline_metrics,
    on=["match_format", "batter_id"],
    how="inner",
    validate="one_to_one"
)

# Strike rate
batter_comparison["pressure_strike_rate"] = (
    batter_comparison["pressure_runs"] * 100
    / batter_comparison["pressure_balls_faced"]
)

batter_comparison["baseline_strike_rate"] = (
    batter_comparison["baseline_runs"] * 100
    / batter_comparison["baseline_balls_faced"]
)

batter_comparison["strike_rate_change"] = (
    batter_comparison["pressure_strike_rate"]
    - batter_comparison["baseline_strike_rate"]
)

batter_comparison["strike_rate_change_pct"] = (
    batter_comparison["strike_rate_change"]
    / batter_comparison["baseline_strike_rate"].replace(0, np.nan)
) * 100

# Batting average
batter_comparison["pressure_average"] = np.where(
    batter_comparison["pressure_dismissals"] > 0,
    batter_comparison["pressure_runs"]
    / batter_comparison["pressure_dismissals"],
    np.nan
)

batter_comparison["baseline_average"] = np.where(
    batter_comparison["baseline_dismissals"] > 0,
    batter_comparison["baseline_runs"]
    / batter_comparison["baseline_dismissals"],
    np.nan
)

batter_comparison["average_change"] = (
    batter_comparison["pressure_average"]
    - batter_comparison["baseline_average"]
)

# Boundary percentage: percentage of balls faced resulting in four or six
batter_comparison["pressure_boundary_pct"] = (
    (batter_comparison["pressure_fours"]
     + batter_comparison["pressure_sixes"]) * 100
    / batter_comparison["pressure_balls_faced"]
)

batter_comparison["baseline_boundary_pct"] = (
    (batter_comparison["baseline_fours"]
     + batter_comparison["baseline_sixes"]) * 100
    / batter_comparison["baseline_balls_faced"]
)

batter_comparison["boundary_pct_change"] = (
    batter_comparison["pressure_boundary_pct"]
    - batter_comparison["baseline_boundary_pct"]
)

# Dismissals per 100 balls faced: lower is better
batter_comparison["pressure_dismissals_per_100_balls"] = (
    batter_comparison["pressure_dismissals"] * 100
    / batter_comparison["pressure_balls_faced"]
)

batter_comparison["baseline_dismissals_per_100_balls"] = (
    batter_comparison["baseline_dismissals"] * 100
    / batter_comparison["baseline_balls_faced"]
)

batter_comparison["dismissal_rate_change"] = (
    batter_comparison["pressure_dismissals_per_100_balls"]
    - batter_comparison["baseline_dismissals_per_100_balls"]
)

# Add player names
batter_comparison = batter_comparison.merge(
    eligible_keys,
    on=["match_format", "batter_id"],
    how="left",
    validate="one_to_one"
)

# Display a compact comparison table
comparison_columns = [
    "match_format", "batter_name",
    "pressure_balls_faced", "baseline_balls_faced",
    "pressure_matches", "baseline_matches",
    "pressure_strike_rate", "baseline_strike_rate",
    "strike_rate_change",
    "pressure_average", "baseline_average",
    "average_change",
    "pressure_boundary_pct", "baseline_boundary_pct",
    "boundary_pct_change",
    "pressure_dismissals_per_100_balls",
    "baseline_dismissals_per_100_balls",
    "dismissal_rate_change"
]

print("BATTER PRESSURE VS BASELINE COMPARISON")
display(
    batter_comparison[comparison_columns]
    .sort_values(["match_format", "strike_rate_change"],
                 ascending=[True, False])
    .round(2)
)

print("\nMETRIC DISTRIBUTION BY FORMAT")
display(
    batter_comparison.groupby("match_format").agg(
        batters=("batter_id", "nunique"),
        median_pressure_sr=("pressure_strike_rate", "median"),
        median_baseline_sr=("baseline_strike_rate", "median"),
        median_sr_change=("strike_rate_change", "median"),
        median_average_change=("average_change", "median"),
        median_boundary_change=("boundary_pct_change", "median"),
        median_dismissal_rate_change=("dismissal_rate_change", "median")
    ).reset_index().round(2)
)

BATTER PRESSURE VS BASELINE COMPARISON


,match_format,batter_name,pressure_balls_faced,baseline_balls_faced,pressure_matches,baseline_matches,pressure_strike_rate,baseline_strike_rate,strike_rate_change,pressure_average,baseline_average,average_change,pressure_boundary_pct,baseline_boundary_pct,boundary_pct_change,pressure_dismissals_per_100_balls,baseline_dismissals_per_100_balls,dismissal_rate_change
3,T20,David Miller,34,481,8,30,264.71,145.74,118.97,45.00,50.07,-5.07,44.12,18.09,26.03,5.88,2.91,2.97
16,T20,Rinku Singh,102,289,7,23,243.14,126.30,116.84,NaN,36.50,NaN,39.22,15.57,23.64,0.00,3.46,-3.46
30,T20,Tristan Stubbs,56,313,6,24,233.93,131.95,101.98,43.67,37.55,6.12,35.71,13.10,22.62,5.36,3.51,1.84
31,T20,Shubham Dubey,23,73,5,10,256.52,157.53,98.99,59.00,28.75,30.25,34.78,23.29,11.49,4.35,5.48,-1.13
24,T20,Sherfane Rutherford,45,210,6,11,217.78,128.57,89.21,32.67,38.57,-5.90,31.11,16.19,14.92,6.67,3.33,3.33
0,T20,Mahendra Singh Dhoni,103,88,13,13,170.87,97.73,73.15,29.33,86.00,-56.67,23.30,7.95,15.35,5.83,1.14,4.69
13,T20,Rahul Tewatia,90,252,14,28,200.00,128.97,71.03,30.00,40.62,-10.62,32.22,17.06,15.16,6.67,3.17,3.49
6,T20,Marcus Stoinis,49,360,8,27,214.29,143.33,70.95,21.00,34.40,-13.40,34.69,20.00,14.69,10.20,4.17,6.04
18,T20,Shivam Dube,45,417,8,26,204.44,133.81,70.63,18.40,39.86,-21.46,28.89,16.55,12.34,11.11,3.36,7.75
17,T20,Rashid Khan,74,83,12,13,213.51,143.37,70.14,31.60,23.80,7.80,33.78,18.07,15.71,6.76,6.02,0.73



METRIC DISTRIBUTION BY FORMAT


,match_format,batters,median_pressure_sr,median_baseline_sr,median_sr_change,median_average_change,median_boundary_change,median_dismissal_rate_change
0,T20,33,192.16,145.74,45.57,-10.63,8.52,3.31
1,T20I,13,163.16,126.09,47.61,-2.33,9.15,2.21


In [21]:
# STEP 19: Audit batting sample reliability

print("BATTER STATS COLUMNS")
print(batter_stats.columns.tolist())

print("\nBATTER DELIVERY DATA COLUMNS")
print(batter_delivery_df.columns.tolist())

print("\nPRESSURE SAMPLE SUMMARY")
display(
    batter_stats[
        batter_stats["performance_group"] == "pressure"
    ].describe(include="all").T
)

print("\nBASELINE SAMPLE SUMMARY")
display(
    batter_stats[
        batter_stats["performance_group"] == "baseline"
    ].describe(include="all").T
)

BATTER STATS COLUMNS
['match_format', 'batter_id', 'batter_name', 'performance_group', 'deliveries', 'balls_faced', 'runs', 'fours', 'sixes', 'matches', 'strike_rate', 'dismissals', 'batting_average']

BATTER DELIVERY DATA COLUMNS
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time', 'comp_id', 'result', 'comp_name', 'match_format', 'season', 'primary_analysis_scope', 'primary_analysis_eligible', 'legal_status', 'legal_ball_min_flag', 'legal_ball_max_flag', 'total_delivery_runs', 'legal_balls_min_after', 'legal_balls_max_after', 'le

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
match_format,549,2,T20I,339,NaN,NaN,NaN,NaN,NaN,NaN,NaN
batter_id,549.0,NaN,NaN,NaN,9638.034608,12004.569469,8.0,2972.0,4904.0,8040.0,67036.0
batter_name,549,452,Dushmantha Chameera,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
performance_group,549,1,pressure,549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
deliveries,549.0,NaN,NaN,NaN,15.588342,18.127235,0.0,4.0,9.0,20.0,115.0
balls_faced,549.0,NaN,NaN,NaN,14.734062,16.942322,0.0,4.0,9.0,19.0,107.0
runs,549.0,NaN,NaN,NaN,22.154827,31.796336,0.0,3.0,11.0,27.0,248.0
fours,549.0,NaN,NaN,NaN,1.606557,2.437173,0.0,0.0,1.0,2.0,19.0
sixes,549.0,NaN,NaN,NaN,1.391621,2.590269,0.0,0.0,0.0,2.0,21.0
matches,549.0,NaN,NaN,NaN,2.595628,2.324194,0.0,1.0,2.0,3.0,14.0



BASELINE SAMPLE SUMMARY


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
match_format,798,2,T20I,533,NaN,NaN,NaN,NaN,NaN,NaN,NaN
batter_id,798.0,NaN,NaN,NaN,9988.12406,12276.631746,8.0,2975.0,5041.0,8221.0,66103.0
batter_name,798,660,Virat Kohli,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
performance_group,798,1,baseline,798,NaN,NaN,NaN,NaN,NaN,NaN,NaN
deliveries,798.0,NaN,NaN,NaN,101.869674,158.104084,0.0,9.0,31.0,128.0,1133.0
balls_faced,798.0,NaN,NaN,NaN,98.209273,152.714321,0.0,8.0,30.0,125.5,1101.0
runs,798.0,NaN,NaN,NaN,134.867168,225.254324,0.0,7.0,33.0,162.25,1590.0
fours,798.0,NaN,NaN,NaN,12.210526,21.942855,0.0,0.0,3.0,14.0,158.0
sixes,798.0,NaN,NaN,NaN,6.045113,10.998481,0.0,0.0,1.0,6.0,67.0
matches,798.0,NaN,NaN,NaN,6.97995,8.100349,0.0,1.0,3.0,9.0,42.0


In [22]:
# STEP 20: Reliability audit of eligible batter candidates only

eligible_keys = eligible_batters[
    ["match_format", "batter_id", "batter_name"]
].drop_duplicates()

eligible_audit = batter_stats.merge(
    eligible_keys,
    on=["match_format", "batter_id", "batter_name"],
    how="inner",
    validate="many_to_one"
)

print("ELIGIBLE PLAYER RECORDS:", len(eligible_keys))
print("\nRECORDS BY PERFORMANCE GROUP:")
display(
    eligible_audit.groupby(
        ["match_format", "performance_group"]
    ).size().unstack(fill_value=0)
)

print("\nELIGIBLE SAMPLE SIZE DISTRIBUTION:")
display(
    eligible_audit.groupby(
        ["match_format", "performance_group"]
    )[
        ["deliveries", "balls_faced", "runs", "dismissals", "matches"]
    ].agg(["min", "median", "max"]).round(2)
)

print("\nZERO-EXPOSURE CHECK:")
display(
    eligible_audit[
        (eligible_audit["balls_faced"] == 0) |
        (eligible_audit["matches"] == 0)
    ][
        [
            "match_format", "batter_name", "performance_group",
            "deliveries", "balls_faced", "runs",
            "dismissals", "matches"
        ]
    ]
)

print("\nMISSING METRIC CHECK:")
display(
    eligible_audit[
        eligible_audit[["strike_rate", "batting_average"]].isna().any(axis=1)
    ][
        [
            "match_format", "batter_name", "performance_group",
            "balls_faced", "runs", "dismissals",
            "strike_rate", "batting_average"
        ]
    ]
)

ELIGIBLE PLAYER RECORDS: 46

RECORDS BY PERFORMANCE GROUP:


performance_group,baseline,pressure
match_format,,
T20,33,33
T20I,13,13



ELIGIBLE SAMPLE SIZE DISTRIBUTION:


deliveries             balls_faced              \
                                      min median  max         min median  max   
match_format performance_group                                                  
T20          baseline                  76  264.0  743          73  252.0  725   
             pressure                  20   52.0  115          19   49.0  107   
T20I         baseline                  83  158.0  545          79  150.0  529   
             pressure                  29   60.0  111          26   57.0  103   

                               runs              dismissals             \
                                min median   max        min median max   
match_format performance_group                                           
T20          baseline            86  347.0  1247          1   11.0  23   
             pressure            25   92.0   248          0    4.0   8   
T20I         baseline            76  206.0   667          0    9.0  25   
             pressure            35   99.0   162          2    3.0   9   

                               matches             
                                   min median max  
match_format performance_group                     
T20          baseline               10   20.0  36  
             pressure                5    7.0  14  
T20I         baseline                7   17.0  36  
             pressure                5    6.0  13


ZERO-EXPOSURE CHECK:


,match_format,batter_name,performance_group,deliveries,balls_faced,runs,dismissals,matches



MISSING METRIC CHECK:


,match_format,batter_name,performance_group,balls_faced,runs,dismissals,strike_rate,batting_average
33,T20,Rinku Singh,pressure,102,248,0,243.137255,NaN
66,T20I,Matthew Wade,baseline,122,169,0,138.524590,NaN


In [23]:
# STEP 21: Build match-level batting performance

match_level_batting = (
    batter_delivery_df[
        batter_delivery_df["performance_group"].isin(["pressure", "baseline"])
    ]
    .groupby(
        [
            "match_format",
            "batter_id",
            "batter_name",
            "match_id",
            "performance_group"
        ],
        as_index=False
    )
    .agg(
        balls_faced=("batter_ball_faced", "sum"),
        runs=("runs", "sum"),
        dismissals=("batter_dismissal", "sum")
    )
)

match_level_batting["match_strike_rate"] = (
    match_level_batting["runs"] /
    match_level_batting["balls_faced"].replace(0, np.nan)
) * 100

# Keep only our eligible candidates
eligible_keys = eligible_batters[
    ["match_format", "batter_id"]
].drop_duplicates()

match_level_batting = match_level_batting.merge(
    eligible_keys,
    on=["match_format", "batter_id"],
    how="inner",
    validate="many_to_one"
)

print("MATCH-LEVEL RECORDS:", len(match_level_batting))

print("\nMATCH-LEVEL SAMPLE DISTRIBUTION:")
display(
    match_level_batting.groupby(
        ["match_format", "performance_group"]
    ).agg(
        player_match_records=("match_id", "size"),
        median_balls_per_match=("balls_faced", "median"),
        min_balls_per_match=("balls_faced", "min"),
        max_balls_per_match=("balls_faced", "max"),
        median_runs_per_match=("runs", "median")
    ).round(2)
)

print("\nPRESSURE MATCH EXPOSURE THRESHOLD AUDIT:")
pressure_match_audit = (
    match_level_batting[
        match_level_batting["performance_group"] == "pressure"
    ]
    .assign(
        at_least_3_balls=lambda x: x["balls_faced"] >= 3,
        at_least_6_balls=lambda x: x["balls_faced"] >= 6,
        at_least_10_balls=lambda x: x["balls_faced"] >= 10
    )
    .groupby("match_format")
    .agg(
        total_pressure_player_matches=("match_id", "size"),
        matches_3plus_balls=("at_least_3_balls", "sum"),
        matches_6plus_balls=("at_least_6_balls", "sum"),
        matches_10plus_balls=("at_least_10_balls", "sum")
    )
)

display(pressure_match_audit)

KeyError: 'batter_id'

In [24]:
# STEP 21 (CORRECTED): Build match-level batting performance

match_level_batting = (
    batter_delivery_df[
        batter_delivery_df["performance_group"].isin(["pressure", "baseline"])
    ]
    .groupby(
        [
            "match_format",
            "striker_id",
            "striker_name",
            "match_id",
            "performance_group"
        ],
        as_index=False
    )
    .agg(
        balls_faced=("batter_ball_faced", "sum"),
        runs=("runs", "sum"),
        dismissals=("batter_dismissal", "sum")
    )
    .rename(columns={
        "striker_id": "batter_id",
        "striker_name": "batter_name"
    })
)

match_level_batting["match_strike_rate"] = (
    match_level_batting["runs"] /
    match_level_batting["balls_faced"].replace(0, np.nan)
) * 100

# Keep only eligible candidates
eligible_keys = eligible_batters[
    ["match_format", "batter_id"]
].drop_duplicates()

match_level_batting = match_level_batting.merge(
    eligible_keys,
    on=["match_format", "batter_id"],
    how="inner",
    validate="many_to_one"
)

print("MATCH-LEVEL RECORDS:", len(match_level_batting))

print("\nMATCH-LEVEL SAMPLE DISTRIBUTION:")
display(
    match_level_batting.groupby(
        ["match_format", "performance_group"]
    ).agg(
        player_match_records=("match_id", "size"),
        median_balls_per_match=("balls_faced", "median"),
        min_balls_per_match=("balls_faced", "min"),
        max_balls_per_match=("balls_faced", "max"),
        median_runs_per_match=("runs", "median")
    ).round(2)
)

print("\nPRESSURE MATCH EXPOSURE THRESHOLD AUDIT:")
pressure_match_audit = (
    match_level_batting[
        match_level_batting["performance_group"] == "pressure"
    ]
    .assign(
        at_least_3_balls=lambda x: x["balls_faced"] >= 3,
        at_least_6_balls=lambda x: x["balls_faced"] >= 6,
        at_least_10_balls=lambda x: x["balls_faced"] >= 10
    )
    .groupby("match_format")
    .agg(
        total_pressure_player_matches=("match_id", "size"),
        matches_3plus_balls=("at_least_3_balls", "sum"),
        matches_6plus_balls=("at_least_6_balls", "sum"),
        matches_10plus_balls=("at_least_10_balls", "sum")
    )
)

display(pressure_match_audit)

MATCH-LEVEL RECORDS: 1275

MATCH-LEVEL SAMPLE DISTRIBUTION:


player_match_records  median_balls_per_match  \
match_format performance_group                                                 
T20          baseline                            699                    10.0   
             pressure                            255                     6.0   
T20I         baseline                            226                    10.0   
             pressure                             95                     6.0   

                                min_balls_per_match  max_balls_per_match  \
match_format performance_group                                             
T20          baseline                             1                   52   
             pressure                             1                   33   
T20I         baseline                             1                   55   
             pressure                             1                   33   

                                median_runs_per_match  
match_format performance_group                         
T20          baseline                            14.0  
             pressure                             9.0  
T20I         baseline                            12.0  
             pressure                             8.0


PRESSURE MATCH EXPOSURE THRESHOLD AUDIT:


,total_pressure_player_matches,matches_3plus_balls,matches_6plus_balls,matches_10plus_balls
match_format,,,,
T20,255,178,134,68
T20I,95,73,52,29


In [25]:
# STEP 22: Calculate match-level pressure consistency

# Get each eligible player's baseline strike rate
baseline_sr_reference = (
    batter_stats[
        batter_stats["performance_group"] == "baseline"
    ][["match_format", "batter_id", "strike_rate"]]
    .rename(columns={"strike_rate": "baseline_strike_rate"})
)

# Attach baseline strike rate to pressure match records
pressure_match_records = match_level_batting[
    match_level_batting["performance_group"] == "pressure"
].merge(
    baseline_sr_reference,
    on=["match_format", "batter_id"],
    how="left",
    validate="many_to_one"
)

consistency_results = []

for min_balls in [3, 6, 10]:

    qualifying = pressure_match_records[
        (pressure_match_records["balls_faced"] >= min_balls) &
        pressure_match_records["baseline_strike_rate"].notna()
    ].copy()

    qualifying["matched_or_better"] = (
        qualifying["match_strike_rate"] >=
        qualifying["baseline_strike_rate"]
    )

    player_consistency = (
        qualifying.groupby(
            ["match_format", "batter_id", "batter_name"],
            as_index=False
        )
        .agg(
            qualifying_pressure_matches=("match_id", "nunique"),
            matches_at_or_above_baseline=("matched_or_better", "sum")
        )
    )

    player_consistency["consistency_pct"] = (
        player_consistency["matches_at_or_above_baseline"] /
        player_consistency["qualifying_pressure_matches"]
    ) * 100

    player_consistency["minimum_balls_threshold"] = min_balls

    consistency_results.append(player_consistency)

consistency_by_threshold = pd.concat(
    consistency_results,
    ignore_index=True
)

# Main consistency result: 6-ball threshold
pressure_consistency = consistency_by_threshold[
    consistency_by_threshold["minimum_balls_threshold"] == 6
].copy()

print("CONSISTENCY RESULTS AT 6-BALL THRESHOLD")
display(
    pressure_consistency.sort_values(
        ["match_format", "consistency_pct"],
        ascending=[True, False]
    ).round(2)
)

print("\nCONSISTENCY THRESHOLD SENSITIVITY")
display(
    consistency_by_threshold.groupby(
        ["match_format", "minimum_balls_threshold"]
    ).agg(
        eligible_batters=("batter_id", "nunique"),
        median_qualifying_matches=("qualifying_pressure_matches", "median"),
        median_consistency_pct=("consistency_pct", "median")
    ).round(2)
)

CONSISTENCY RESULTS AT 6-BALL THRESHOLD


,match_format,batter_id,batter_name,qualifying_pressure_matches,matches_at_or_above_baseline,consistency_pct,minimum_balls_threshold
47,T20,143,Dinesh Karthik,3,3,100.00,6
49,T20,1077,David Miller,2,2,100.00,6
51,T20,1699,Suryakumar Yadav,1,1,100.00,6
52,T20,2639,Marcus Stoinis,4,4,100.00,6
56,T20,3297,Shimron Hetmyer,3,3,100.00,6
58,T20,3782,M Shahrukh Khan,2,2,100.00,6
62,T20,4708,Rinku Singh,7,7,100.00,6
63,T20,4722,Rashid Khan,5,5,100.00,6
64,T20,4755,Shivam Dube,3,3,100.00,6
67,T20,6396,Tim David,4,4,100.00,6



CONSISTENCY THRESHOLD SENSITIVITY


eligible_batters  \
match_format minimum_balls_threshold                     
T20          3                                      33   
             6                                      33   
             10                                     29   
T20I         3                                      13   
             6                                      13   
             10                                     11   

                                      median_qualifying_matches  \
match_format minimum_balls_threshold                              
T20          3                                              5.0   
             6                                              4.0   
             10                                             2.0   
T20I         3                                              5.0   
             6                                              4.0   
             10                                             2.0   

                                      median_consistency_pct  
match_format minimum_balls_threshold                          
T20          3                                         75.00  
             6                                         88.89  
             10                                       100.00  
T20I         3                                         60.00  
             6                                         71.43  
             10                                       100.00

In [26]:
# STEP 23: Sample-size-adjusted pressure consistency

import numpy as np

consistency_reliability = pressure_consistency.copy()

z = 1.96  # 95% Wilson confidence level

n = consistency_reliability["qualifying_pressure_matches"]
successes = consistency_reliability["matches_at_or_above_baseline"]

p = successes / n

denominator = 1 + (z**2 / n)

center = p + (z**2 / (2 * n))

margin = z * np.sqrt(
    (p * (1 - p) / n) +
    (z**2 / (4 * n**2))
)

consistency_reliability["wilson_lower_95"] = (
    (center - margin) / denominator
) * 100

consistency_reliability["wilson_lower_95"] = (
    consistency_reliability["wilson_lower_95"].round(2)
)

print("SAMPLE-SIZE-ADJUSTED CONSISTENCY")
display(
    consistency_reliability.sort_values(
        ["match_format", "wilson_lower_95"],
        ascending=[True, False]
    )[
        [
            "match_format",
            "batter_name",
            "qualifying_pressure_matches",
            "matches_at_or_above_baseline",
            "consistency_pct",
            "wilson_lower_95"
        ]
    ]
)

print("\nRELIABILITY SUMMARY")
display(
    consistency_reliability.groupby("match_format").agg(
        eligible_batters=("batter_id", "nunique"),
        median_qualifying_matches=("qualifying_pressure_matches", "median"),
        median_raw_consistency=("consistency_pct", "median"),
        median_wilson_lower=("wilson_lower_95", "median")
    ).round(2)
)

SAMPLE-SIZE-ADJUSTED CONSISTENCY


,match_format,batter_name,qualifying_pressure_matches,matches_at_or_above_baseline,consistency_pct,wilson_lower_95
62,T20,Rinku Singh,7,7,100.000000,64.57
63,T20,Rashid Khan,5,5,100.000000,56.55
72,T20,Dhruv Jurel,5,5,100.000000,56.55
74,T20,Namboori Tilak Varma,5,5,100.000000,56.55
76,T20,Tristan Stubbs,5,5,100.000000,56.55
46,T20,Mahendra Singh Dhoni,9,8,88.888889,56.50
48,T20,Ravindra Jadeja,8,7,87.500000,52.91
52,T20,Marcus Stoinis,4,4,100.000000,51.01
67,T20,Tim David,4,4,100.000000,51.01
47,T20,Dinesh Karthik,3,3,100.000000,43.85



RELIABILITY SUMMARY


,eligible_batters,median_qualifying_matches,median_raw_consistency,median_wilson_lower
match_format,,,,
T20,33,4.0,88.89,34.24
T20I,13,4.0,71.43,30.06


In [27]:
# STEP 24: Consolidate all batter performance metrics

ranking_data = batter_comparison.copy()

# Add pressure consistency and reliability
ranking_data = ranking_data.merge(
    consistency_reliability[
        [
            "match_format",
            "batter_id",
            "qualifying_pressure_matches",
            "matches_at_or_above_baseline",
            "consistency_pct",
            "wilson_lower_95"
        ]
    ],
    on=["match_format", "batter_id"],
    how="left",
    validate="one_to_one"
)

# Add pressure and baseline run totals
ranking_data["pressure_runs"] = ranking_data["pressure_runs"]
ranking_data["baseline_runs"] = ranking_data["baseline_runs"]

# Add eligibility sample information
ranking_data["pressure_sample_eligible"] = (
    (ranking_data["pressure_balls_faced"] >= 18) &
    (ranking_data["pressure_matches"] >= 5)
)

ranking_data["baseline_sample_eligible"] = (
    (ranking_data["baseline_balls_faced"] >= 60) &
    (ranking_data["baseline_matches"] >= 3)
)

print("RANKING DATA SHAPE:", ranking_data.shape)

print("\nMISSING VALUES:")
display(
    ranking_data.isna().sum().to_frame("missing_values")
)

print("\nCONSOLIDATED RANKING INPUT:")
display(
    ranking_data.sort_values(
        ["match_format", "pressure_strike_rate"],
        ascending=[True, False]
    ).round(2)
)

RANKING DATA SHAPE: (46, 36)

MISSING VALUES:


,missing_values
match_format,0
batter_id,0
pressure_deliveries,0
pressure_balls_faced,0
pressure_runs,0
pressure_fours,0
pressure_sixes,0
pressure_dismissals,0
pressure_matches,0
baseline_deliveries,0



CONSOLIDATED RANKING INPUT:


,match_format,batter_id,pressure_deliveries,pressure_balls_faced,pressure_runs,pressure_fours,pressure_sixes,pressure_dismissals,pressure_matches,baseline_deliveries,...,pressure_dismissals_per_100_balls,baseline_dismissals_per_100_balls,dismissal_rate_change,batter_name,qualifying_pressure_matches,matches_at_or_above_baseline,consistency_pct,wilson_lower_95,pressure_sample_eligible,baseline_sample_eligible
3,T20,1077.0,36,34,90,6,9,2,8,502,...,5.88,2.91,2.97,David Miller,2,2,100.00,34.24,True,True
31,T20,25139.0,23,23,59,2,6,1,5,76,...,4.35,5.48,-1.13,Shubham Dubey,2,2,100.00,34.24,True,True
16,T20,4708.0,112,102,248,19,21,0,7,302,...,0.00,3.46,-3.46,Rinku Singh,7,7,100.00,64.57,True,True
30,T20,20702.0,58,56,131,10,10,3,6,321,...,5.36,3.51,1.84,Tristan Stubbs,5,5,100.00,56.55,True,True
1,T20,143.0,70,57,129,7,12,6,7,138,...,10.53,3.03,7.50,Dinesh Karthik,3,3,100.00,43.85,True,True
21,T20,6396.0,87,77,168,9,16,6,12,225,...,7.79,5.16,2.63,Tim David,4,4,100.00,51.01,True,True
24,T20,6725.0,51,45,98,5,9,3,6,224,...,6.67,3.33,3.33,Sherfane Rutherford,3,3,100.00,43.85,True,True
29,T20,8101.0,35,30,65,5,5,3,5,178,...,10.00,4.05,5.95,Ashutosh Sharma,2,1,50.00,9.45,True,True
26,T20,7260.0,67,61,131,11,10,3,6,361,...,4.92,4.52,0.40,Dhruv Jurel,5,5,100.00,56.55,True,True
6,T20,2639.0,52,49,105,8,9,5,8,382,...,10.20,4.17,6.04,Marcus Stoinis,4,4,100.00,51.01,True,True


In [28]:
# STEP 25: Metric relationship and distribution audit

metrics_for_audit = [
    "strike_rate_change",
    "boundary_pct_change",
    "dismissal_rate_change",
    "wilson_lower_95",
    "pressure_balls_faced",
    "baseline_balls_faced",
    "pressure_runs"
]

print("METRIC CORRELATION MATRIX")
display(
    ranking_data[metrics_for_audit]
    .corr(method="spearman")
    .round(2)
)

print("\nMETRIC DISTRIBUTION BY FORMAT")
display(
    ranking_data.groupby("match_format")[metrics_for_audit]
    .agg(["median", "mean", "std", "min", "max"])
    .round(2)
)

print("\nPLAYER RECORD COUNTS")
display(
    ranking_data.groupby("match_format").agg(
        player_records=("batter_id", "count"),
        unique_players=("batter_id", "nunique"),
        total_pressure_runs=("pressure_runs", "sum"),
        total_pressure_balls=("pressure_balls_faced", "sum")
    )
)

METRIC CORRELATION MATRIX


,strike_rate_change,boundary_pct_change,dismissal_rate_change,wilson_lower_95,pressure_balls_faced,baseline_balls_faced,pressure_runs
strike_rate_change,1.00,0.93,-0.35,0.71,0.23,0.01,0.51
boundary_pct_change,0.93,1.00,-0.33,0.68,0.26,0.07,0.52
dismissal_rate_change,-0.35,-0.33,1.00,-0.26,-0.46,0.16,-0.49
wilson_lower_95,0.71,0.68,-0.26,1.00,0.48,0.14,0.67
pressure_balls_faced,0.23,0.26,-0.46,0.48,1.00,-0.00,0.93
baseline_balls_faced,0.01,0.07,0.16,0.14,-0.00,1.00,0.05
pressure_runs,0.51,0.52,-0.49,0.67,0.93,0.05,1.00



METRIC DISTRIBUTION BY FORMAT


strike_rate_change                               \
                         median   mean    std    min     max   
match_format                                                   
T20                       45.57  44.61  40.13 -40.42  118.97   
T20I                      47.61  47.96  36.36  -5.46  117.08   

             boundary_pct_change                           ...  \
                          median  mean   std   min    max  ...   
match_format                                               ...   
T20                         8.52  8.19  8.11 -5.15  26.03  ...   
T20I                        9.15  9.43  8.27 -2.52  25.31  ...   

             baseline_balls_faced                          pressure_runs  \
                           median    mean     std min  max        median   
match_format                                                               
T20                         252.0  269.48  153.41  73  725          92.0   
T20I                        150.0  214.31  147.75  79  529          99.0   

                                      
                mean    std min  max  
match_format                          
T20           102.88  50.85  25  248  
T20I           96.69  41.99  35  162  

[2 rows x 35 columns]


PLAYER RECORD COUNTS


,player_records,unique_players,total_pressure_runs,total_pressure_balls
match_format,,,,
T20,33,33,3395,1791
T20I,13,13,1257,722


In [29]:
# STEP 26: Format-specific metric correlations

format_correlations = {}

for fmt in ["T20", "T20I"]:
    format_data = ranking_data[
        ranking_data["match_format"] == fmt
    ]

    format_correlations[fmt] = format_data[
        [
            "strike_rate_change",
            "boundary_pct_change",
            "dismissal_rate_change",
            "wilson_lower_95",
            "pressure_balls_faced",
            "baseline_balls_faced"
        ]
    ].corr(method="spearman")

    print(f"\n{fmt} SPEARMAN CORRELATION")
    display(format_correlations[fmt].round(2))


T20 SPEARMAN CORRELATION


,strike_rate_change,boundary_pct_change,dismissal_rate_change,wilson_lower_95,pressure_balls_faced,baseline_balls_faced
strike_rate_change,1.00,0.93,-0.26,0.73,0.22,0.01
boundary_pct_change,0.93,1.00,-0.26,0.75,0.27,0.11
dismissal_rate_change,-0.26,-0.26,1.00,-0.29,-0.49,0.25
wilson_lower_95,0.73,0.75,-0.29,1.00,0.54,0.18
pressure_balls_faced,0.22,0.27,-0.49,0.54,1.00,-0.05
baseline_balls_faced,0.01,0.11,0.25,0.18,-0.05,1.00



T20I SPEARMAN CORRELATION


,strike_rate_change,boundary_pct_change,dismissal_rate_change,wilson_lower_95,pressure_balls_faced,baseline_balls_faced
strike_rate_change,1.00,0.96,-0.54,0.73,0.30,0.15
boundary_pct_change,0.96,1.00,-0.49,0.62,0.20,0.05
dismissal_rate_change,-0.54,-0.49,1.00,-0.28,-0.47,-0.12
wilson_lower_95,0.73,0.62,-0.28,1.00,0.38,-0.09
pressure_balls_faced,0.30,0.20,-0.47,0.38,1.00,0.23
baseline_balls_faced,0.15,0.05,-0.12,-0.09,0.23,1.00


In [30]:
# STEP 27: Build transparent preliminary batter ranking scores

scored_batters = ranking_data.copy()

# Convert performance metrics into within-format percentile scores.
# Higher strike-rate improvement is better.
scored_batters["sr_percentile"] = (
    scored_batters.groupby("match_format")["strike_rate_change"]
    .rank(pct=True) * 100
)

# Lower dismissal-rate change is better.
scored_batters["dismissal_control_percentile"] = (
    scored_batters.groupby("match_format")["dismissal_rate_change"]
    .rank(ascending=False, pct=True) * 100
)

# Primary methodology: 60% scoring improvement, 40% dismissal control.
scored_batters["composite_score_60_40"] = (
    0.60 * scored_batters["sr_percentile"] +
    0.40 * scored_batters["dismissal_control_percentile"]
)

# Alternative weighting scenarios for sensitivity analysis.
scored_batters["composite_score_50_50"] = (
    0.50 * scored_batters["sr_percentile"] +
    0.50 * scored_batters["dismissal_control_percentile"]
)

scored_batters["composite_score_70_30"] = (
    0.70 * scored_batters["sr_percentile"] +
    0.30 * scored_batters["dismissal_control_percentile"]
)

# Wilson lower bound is used only as a supporting reliability indicator.
scored_batters = scored_batters.sort_values(
    ["match_format", "composite_score_60_40", "wilson_lower_95"],
    ascending=[True, False, False]
)

print("PRELIMINARY BATTER RANKING SCORES")

display(
    scored_batters[
        [
            "match_format",
            "batter_name",
            "pressure_balls_faced",
            "pressure_matches",
            "strike_rate_change",
            "dismissal_rate_change",
            "sr_percentile",
            "dismissal_control_percentile",
            "composite_score_60_40",
            "wilson_lower_95"
        ]
    ].round(2)
)

print("\nTOP 10 UNDER EACH FORMAT — PRELIMINARY 60:40 MODEL")

for fmt in ["T20", "T20I"]:
    print(f"\n{fmt}")
    display(
        scored_batters[
            scored_batters["match_format"] == fmt
        ].head(10)[
            [
                "batter_name",
                "composite_score_60_40",
                "strike_rate_change",
                "dismissal_rate_change",
                "pressure_balls_faced",
                "qualifying_pressure_matches",
                "wilson_lower_95"
            ]
        ].round(2)
    )

PRELIMINARY BATTER RANKING SCORES


,match_format,batter_name,pressure_balls_faced,pressure_matches,strike_rate_change,dismissal_rate_change,sr_percentile,dismissal_control_percentile,composite_score_60_40,wilson_lower_95
16,T20,Rinku Singh,102,7,116.84,-3.46,96.97,96.97,96.97,64.57
31,T20,Shubham Dubey,23,5,98.99,-1.13,90.91,90.91,90.91,34.24
30,T20,Tristan Stubbs,56,6,101.98,1.84,93.94,69.70,84.24,56.55
3,T20,David Miller,34,8,118.97,2.97,100.00,54.55,81.82,34.24
17,T20,Rashid Khan,74,12,70.14,0.73,72.73,81.82,76.36,56.55
26,T20,Dhruv Jurel,61,6,67.30,0.40,63.64,84.85,72.12,56.55
24,T20,Sherfane Rutherford,45,6,89.21,3.33,87.88,48.48,72.12,43.85
22,T20,Ayush Badoni,49,5,63.31,-0.17,60.61,87.88,71.52,20.77
2,T20,Ravindra Jadeja,107,13,51.29,-1.94,54.55,93.94,70.30,52.91
0,T20,Mahendra Singh Dhoni,103,13,73.15,4.69,84.85,42.42,67.88,56.50



TOP 10 UNDER EACH FORMAT — PRELIMINARY 60:40 MODEL

T20


,batter_name,composite_score_60_40,strike_rate_change,dismissal_rate_change,pressure_balls_faced,qualifying_pressure_matches,wilson_lower_95
16,Rinku Singh,96.97,116.84,-3.46,102,7,64.57
31,Shubham Dubey,90.91,98.99,-1.13,23,2,34.24
30,Tristan Stubbs,84.24,101.98,1.84,56,5,56.55
3,David Miller,81.82,118.97,2.97,34,2,34.24
17,Rashid Khan,76.36,70.14,0.73,74,5,56.55
26,Dhruv Jurel,72.12,67.30,0.40,61,5,56.55
24,Sherfane Rutherford,72.12,89.21,3.33,45,3,43.85
22,Ayush Badoni,71.52,63.31,-0.17,49,3,20.77
2,Ravindra Jadeja,70.30,51.29,-1.94,107,8,52.91
0,Mahendra Singh Dhoni,67.88,73.15,4.69,103,9,56.50



T20I


,batter_name,composite_score_60_40,strike_rate_change,dismissal_rate_change,pressure_balls_faced,qualifying_pressure_matches,wilson_lower_95
38,Jason Holder,90.77,80.16,-5.56,84,7,35.89
41,Rovman Powell,89.23,82.71,0.22,74,5,56.55
36,Dasun Shanaka,78.46,117.08,2.60,40,3,43.85
39,Mitchell Santner,72.31,73.97,0.87,58,4,30.06
42,Mark Adair,69.23,47.61,-5.00,39,4,30.06
34,David Miller,61.54,50.38,1.94,61,4,15.00
45,Namboori Tilak Varma,55.38,42.17,1.28,63,3,20.77
35,Akeal Hosein,52.31,74.79,7.63,57,4,51.01
37,Ryan Burl,44.62,23.49,2.21,42,2,9.45
33,Matthew Wade,30.77,19.25,4.44,45,3,43.85


In [31]:
# STEP 28: Ranking sensitivity and Top-10 stability

from scipy.stats import spearmanr
import pandas as pd

score_columns = {
    "50:50": "composite_score_50_50",
    "60:40": "composite_score_60_40",
    "70:30": "composite_score_70_30"
}

sensitivity_results = []
top10_membership = []

for fmt in ["T20", "T20I"]:

    format_data = scored_batters[
        scored_batters["match_format"] == fmt
    ].copy()

    # Calculate ranks independently for each weighting scenario.
    for scenario, score_col in score_columns.items():
        format_data[f"rank_{scenario}"] = (
            format_data[score_col]
            .rank(method="min", ascending=False)
            .astype(int)
        )

        top10_names = set(
            format_data.nsmallest(10, f"rank_{scenario}")["batter_name"]
        )

        for player in top10_names:
            top10_membership.append({
                "match_format": fmt,
                "scenario": scenario,
                "batter_name": player
            })

    # Compare ranking correlations.
    rank_columns = [
        f"rank_{scenario}" for scenario in score_columns
    ]

    rank_corr = format_data[rank_columns].corr(method="spearman")

    print(f"\n{fmt} — RANK CORRELATION BETWEEN WEIGHTING SCENARIOS")
    display(rank_corr.round(3))

    # Compare Top-10 overlap.
    top10_sets = {
        scenario: set(
            format_data.nsmallest(10, f"rank_{scenario}")["batter_name"]
        )
        for scenario in score_columns
    }

    print(f"\n{fmt} — TOP-10 OVERLAP")

    for first, second in [
        ("50:50", "60:40"),
        ("60:40", "70:30"),
        ("50:50", "70:30")
    ]:
        overlap = top10_sets[first] & top10_sets[second]

        sensitivity_results.append({
            "match_format": fmt,
            "comparison": f"{first} vs {second}",
            "overlap_count": len(overlap),
            "overlap_players": ", ".join(sorted(overlap))
        })

display(pd.DataFrame(sensitivity_results))

print("\nTOP-10 MEMBERSHIP ACROSS ALL SCENARIOS")
membership_df = pd.DataFrame(top10_membership)

membership_summary = (
    membership_df.groupby(["match_format", "batter_name"])
    ["scenario"]
    .nunique()
    .reset_index(name="scenarios_in_top10")
    .sort_values(
        ["match_format", "scenarios_in_top10", "batter_name"],
        ascending=[True, False, True]
    )
)

display(membership_summary)


T20 — RANK CORRELATION BETWEEN WEIGHTING SCENARIOS


,rank_50:50,rank_60:40,rank_70:30
rank_50:50,1.000,0.986,0.952
rank_60:40,0.986,1.000,0.984
rank_70:30,0.952,0.984,1.000



T20 — TOP-10 OVERLAP

T20I — RANK CORRELATION BETWEEN WEIGHTING SCENARIOS


,rank_50:50,rank_60:40,rank_70:30
rank_50:50,1.000,0.993,0.974
rank_60:40,0.993,1.000,0.984
rank_70:30,0.974,0.984,1.000



T20I — TOP-10 OVERLAP


,match_format,comparison,overlap_count,overlap_players
0,T20,50:50 vs 60:40,9,"Ayush Badoni, David Miller, Dhruv Jurel, Rashi..."
1,T20,60:40 vs 70:30,9,"Ayush Badoni, David Miller, Dhruv Jurel, Mahen..."
2,T20,50:50 vs 70:30,8,"Ayush Badoni, David Miller, Dhruv Jurel, Rashi..."
3,T20I,50:50 vs 60:40,10,"Akeal Hosein, Dasun Shanaka, David Miller, Jas..."
4,T20I,60:40 vs 70:30,10,"Akeal Hosein, Dasun Shanaka, David Miller, Jas..."
5,T20I,50:50 vs 70:30,10,"Akeal Hosein, Dasun Shanaka, David Miller, Jas..."



TOP-10 MEMBERSHIP ACROSS ALL SCENARIOS


,match_format,batter_name,scenarios_in_top10
0,T20,Ayush Badoni,3
1,T20,David Miller,3
2,T20,Dhruv Jurel,3
6,T20,Rashid Khan,3
8,T20,Rinku Singh,3
9,T20,Sherfane Rutherford,3
10,T20,Shubham Dubey,3
11,T20,Tristan Stubbs,3
3,T20,Mahendra Singh Dhoni,2
7,T20,Ravindra Jadeja,2


In [32]:
# STEP 29: Final preliminary batter Top-10 tables (60:40)

final_batter_top10 = (
    scored_batters
    .sort_values(
        ["match_format", "composite_score_60_40", "wilson_lower_95"],
        ascending=[True, False, False]
    )
    .groupby("match_format", group_keys=False)
    .head(10)
    .copy()
)

display_columns = [
    "match_format",
    "batter_name",
    "composite_score_60_40",
    "pressure_strike_rate",
    "baseline_strike_rate",
    "strike_rate_change",
    "dismissal_rate_change",
    "pressure_balls_faced",
    "pressure_matches",
    "qualifying_pressure_matches",
    "wilson_lower_95"
]

final_batter_top10 = final_batter_top10[display_columns].reset_index(drop=True)

print("FINAL PRELIMINARY BATTER TOP 10 — 60:40 SCORING")
display(final_batter_top10)

print("\nPLAYER COUNTS")
print(final_batter_top10.groupby("match_format")["batter_name"].count())

print("\nQUALIFYING PRESSURE MATCHES — SAMPLE AUDIT")
display(
    final_batter_top10.groupby("match_format")[
        ["pressure_balls_faced", "pressure_matches",
         "qualifying_pressure_matches", "wilson_lower_95"]
    ].describe().round(2)
)

FINAL PRELIMINARY BATTER TOP 10 — 60:40 SCORING


,match_format,batter_name,composite_score_60_40,pressure_strike_rate,baseline_strike_rate,strike_rate_change,dismissal_rate_change,pressure_balls_faced,pressure_matches,qualifying_pressure_matches,wilson_lower_95
0,T20,Rinku Singh,96.969697,243.137255,126.297578,116.839677,-3.460208,102,7,7,64.57
1,T20,Shubham Dubey,90.909091,256.521739,157.534247,98.987493,-1.131626,23,5,2,34.24
2,T20,Tristan Stubbs,84.242424,233.928571,131.948882,101.979690,1.842766,56,6,5,56.55
3,T20,David Miller,81.818182,264.705882,145.738046,118.967837,2.971750,34,8,2,34.24
4,T20,Rashid Khan,76.363636,213.513514,143.373494,70.140020,0.732660,74,12,5,56.55
5,T20,Dhruv Jurel,72.121212,214.754098,147.457627,67.296471,0.398259,61,6,5,56.55
6,T20,Sherfane Rutherford,72.121212,217.777778,128.571429,89.206349,3.333333,45,6,3,43.85
7,T20,Ayush Badoni,71.515152,187.755102,124.444444,63.310658,-0.173847,49,5,3,20.77
8,T20,Ravindra Jadeja,70.303030,172.897196,121.610169,51.287027,-1.944400,107,13,8,52.91
9,T20,Mahendra Singh Dhoni,67.878788,170.873786,97.727273,73.146514,4.688879,103,13,9,56.50



PLAYER COUNTS
match_format
T20     10
T20I    10
Name: batter_name, dtype: int64

QUALIFYING PRESSURE MATCHES — SAMPLE AUDIT


pressure_balls_faced                                        \
                            count  mean    std   min    25%   50%   75%   
match_format                                                              
T20                          10.0  65.4  30.06  23.0  46.00  58.5  95.0   
T20I                         10.0  56.3  15.09  39.0  42.75  57.5  62.5   

                    pressure_matches       ... qualifying_pressure_matches  \
                max            count mean  ...                         75%   
match_format                               ...                               
T20           107.0             10.0  8.1  ...                         6.5   
T20I           84.0             10.0  6.9  ...                         4.0   

                  wilson_lower_95                                            \
              max           count   mean    std    min    25%    50%    75%   
match_format                                                                  
T20           9.0            10.0  47.67  13.85  20.77  36.64  54.70  56.55   
T20I          7.0            10.0  33.65  15.51   9.45  23.09  32.98  43.85   

                     
                max  
match_format         
T20           64.57  
T20I          56.55  

[2 rows x 32 columns]

In [33]:
# STEP 30: Add sample-reliability labels to preliminary Top-10 rankings

def classify_pressure_reliability(matches):
    if matches >= 5:
        return "Strong"
    elif matches >= 3:
        return "Moderate"
    else:
        return "Limited"


final_batter_top10["sample_reliability"] = (
    final_batter_top10["qualifying_pressure_matches"]
    .apply(classify_pressure_reliability)
)

print("TOP-10 BATTER RANKING WITH SAMPLE RELIABILITY")

display(
    final_batter_top10[
        [
            "match_format",
            "batter_name",
            "composite_score_60_40",
            "strike_rate_change",
            "dismissal_rate_change",
            "pressure_balls_faced",
            "qualifying_pressure_matches",
            "wilson_lower_95",
            "sample_reliability"
        ]
    ].round(2)
)

print("\nRELIABILITY DISTRIBUTION")

display(
    final_batter_top10.groupby(
        ["match_format", "sample_reliability"]
    ).size().unstack(fill_value=0)
)

TOP-10 BATTER RANKING WITH SAMPLE RELIABILITY


,match_format,batter_name,composite_score_60_40,strike_rate_change,dismissal_rate_change,pressure_balls_faced,qualifying_pressure_matches,wilson_lower_95,sample_reliability
0,T20,Rinku Singh,96.97,116.84,-3.46,102,7,64.57,Strong
1,T20,Shubham Dubey,90.91,98.99,-1.13,23,2,34.24,Limited
2,T20,Tristan Stubbs,84.24,101.98,1.84,56,5,56.55,Strong
3,T20,David Miller,81.82,118.97,2.97,34,2,34.24,Limited
4,T20,Rashid Khan,76.36,70.14,0.73,74,5,56.55,Strong
5,T20,Dhruv Jurel,72.12,67.30,0.40,61,5,56.55,Strong
6,T20,Sherfane Rutherford,72.12,89.21,3.33,45,3,43.85,Moderate
7,T20,Ayush Badoni,71.52,63.31,-0.17,49,3,20.77,Moderate
8,T20,Ravindra Jadeja,70.30,51.29,-1.94,107,8,52.91,Strong
9,T20,Mahendra Singh Dhoni,67.88,73.15,4.69,103,9,56.50,Strong



RELIABILITY DISTRIBUTION


sample_reliability,Limited,Moderate,Strong
match_format,,,
T20,2,2,6
T20I,1,7,2


In [34]:
# STEP 31: Save preliminary batter ranking outputs

from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save final Top-10 table
final_batter_top10.to_csv(
    output_dir / "preliminary_batter_top10.csv",
    index=False
)

# Save all eligible batters with their scores
scored_batters.to_csv(
    output_dir / "all_eligible_batter_scores.csv",
    index=False
)

print("Batter analysis outputs saved successfully.")
print("Files created:")
print(output_dir / "preliminary_batter_top10.csv")
print(output_dir / "all_eligible_batter_scores.csv")

print("\nSaved row counts:")
print("Top-10 ranking:", len(final_batter_top10))
print("All eligible batters:", len(scored_batters))

Batter analysis outputs saved successfully.
Files created:
..\data\processed\preliminary_batter_top10.csv
..\data\processed\all_eligible_batter_scores.csv

Saved row counts:
Top-10 ranking: 20
All eligible batters: 46
